# Project 3 — notebook gộp: `du-lieu-do-thi` + pipeline ảnh/video

Notebook này gộp hai pipeline của Project 3 (Truy vấn đa phương thức có giải thích bằng đồ thị) để chạy một lần trên Kaggle.

| Phần | Nguồn | Nội dung | Thư mục kết quả |
|---|---|---|---|
| **A** | `hieutrinh123/du-lieu-do-thi` (version 9/11) | Flickr8k (text → ảnh) và Action Genome (text → video): CLIP → đồ thị cảnh (node đối tượng, quan hệ, hành động/bối cảnh Charades) → GNN + đường đi + nhánh khớp truy vấn↔node → xếp hạng lại; giải thích, đối chứng, kiểm định cặp. Phần 4 (MSVD → MSR-VTT) cũ vẫn tắt (`%%script true`). | `/kaggle/working/results/`, `/kaggle/working/output/action_genome/results/` |
| **C** | `lamvo30ai/project03-level4b-v2-graphfix` (version 1), mục 4u trong phần B | Text → video MSVD + MSR-VTT: đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học, ConceptNet, xAI; tạo cache đặc trưng frame cho mục 4v. Chạy trong tiến trình riêng (papermill). | `/kaggle/working/project03_level4b_v2/`, `/kaggle/working/p3_out/video_v2/` |
| **B** | `lamvo30ai/project3-full-pipeline` (version 8) | Visual Genome + COCO (text → ảnh, scene graph GT; GraphSAGE; SCENIR), ARO + SugarCrepe (ảnh → caption, RelTR), COCO-spatial (cạnh hình học), XAI, video MSVD (Level4B v5). | `/kaggle/working/p3_out/` |

**Kaggle:** GPU T4, Internet On. Datasets: `hieutrinh123/project3-merged-assets` (payload nhúng của phần B và notebook phần C, tách ra vì Kaggle giới hạn mã nguồn 1 MB), `hieutrinh123/action-genome`, `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`. Notebook nguồn: `hieutrinh123/du-lieu-do-thi` (cache CLIP), `lamvo30ai/project03-level4b-v2-graphfix` (cache video của phần B — nếu không có quyền truy cập, nhánh video của phần B ghi lỗi và phần còn lại vẫn chạy).

**Thời gian ước tính:** phần A khoảng 4–5 giờ (ViT-L/14), phần B khoảng 3 giờ ⇒ 7–8 giờ. Phần C: gần như 0 nếu output `lamvo30ai/project03-level4b-v2-graphfix` đã được Add Input (chế độ `auto` chỉ chép kết quả), ngược lại +2,4 giờ (tổng ~10 giờ, sát giới hạn 12 giờ của Kaggle).

**Thay đổi khi gộp:** ngoài việc nhúng phần C (mục 4u), chỉ hai chỗ — (1) phần A đặt backbone ViT-L/14 như mô hình cuối (version 9); (2) phần B đổi tên hàm `log` thành `p3_log` để không đè hàm `log` của phần A. Ngoài ra thêm một cell giải phóng bộ nhớ GPU giữa hai phần. Không đổi logic, tham số hay dữ liệu của phần nào.

**Bản Kaggle:** đã bỏ các cell bị tắt sẵn (`%%script true`, gồm Phần 4 MSVD → MSR-VTT cũ) và chuyển hai payload lớn sang dataset để mã nguồn dưới 1 MB; logic chạy không đổi.


> **Lần chạy này chỉ gồm phần C và phần B.** Phần A đã chạy xong ở version 1 của kernel này; kết quả lưu trong `results/project3-merged-v1/` trên GitHub. Code phần B/C giống hệt `Project3_Merged_kaggle.ipynb`, kèm bản sửa `find_inputs`.


---
# PHẦN B — pipeline ảnh + video (Visual Genome, ARO, SugarCrepe, COCO-spatial, MSVD)


# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị (notebook final)

Notebook tự chứa, chạy một lần từ đầu đến cuối trên Kaggle (GPU T4, khoảng 3 giờ: video ~2–2,5 giờ + ảnh ~30 phút), bao phủ các yêu cầu cấp 1, 2, 3 (phần giải thích) và cấp 4 hướng (a), (b) của đề cương.
**Ngoài phạm vi phần B:** bộ 20 truy vấn gián tiếp nằm ở phần A; MMKG tự xây nằm ở `mmkg/` (notebook riêng).

| Mục | Nội dung | Yêu cầu đề cương |
|---|---|---|
| 1–5 | Thiết lập · dữ liệu nhúng · lõi đồ thị · dữ liệu + luật chống rò rỉ · mô hình đóng băng (CLIP ViT-L/14, RelTR) | Mục 4, 6 |
| 4v | **Video** MSVD (text-to-video): notebook Level4B v5 — CLIP + PE-Core + đồ thị bằng chứng (hub, cầu nối) + Qwen2.5-VL | Cấp 4 (b) |
| 6 | **TN1** văn bản → ảnh, full-gallery: CLIP top-100 → điểm đồ thị (đường đi trên scene graph VG) → xếp hạng lại α | Cấp 1 (nhánh A) |
| 6b | **TN1b** GraphSAGE (GNN đã học) làm điểm đồ thị, cùng pipeline; ablation cạnh ngẫu nhiên / bỏ cạnh | Cấp 1 (thuật toán đã học) |
| 6c | **TN1c** SCENIR (ICML 2025), Graph Autoencoder không giám sát, cùng pipeline; ablation | Cấp 2 (phương pháp từ 2024) |
| 7 | **TN2** ARO VG-Relation + SugarCrepe: ảnh → caption đúng, scene graph RelTR từ điểm ảnh | Cấp 2 (≥ 2 tập), vai trò đồ thị |
| 7b | **TN2b** cạnh hình học suy từ box trên COCO-spatial (What'sUp) | Cấp 4 (a): tập công khai thứ hai |
| 8 | **XAI**: fidelity (xoá bằng chứng, lật ảnh), validity (khớp GT), sufficiency, sparsity + 10 ví dụ | Cấp 3 |
| 10 | Demo `search()`: nhập truy vấn → top-k ảnh + quan hệ giải thích | Mục 8 |
| 10b | 5 truy vấn thành công + 5 thất bại (văn bản → ảnh), luật chọn cố định | Cấp 1 |
| 11a | Độ nhạy α · so sánh cách tạo cạnh · xuất file split | Cấp 2 (mở rộng), mục 8 |
| 11 | Bảng tổng hợp · đối chiếu tái lập · **bảng đối chiếu yêu cầu đề cương** | — |

**Nguyên tắc chống rò rỉ:** không huấn luyện CLIP / RelTR; GNN (GraphSAGE, SCENIR) chỉ học trên ảnh không thuộc tập đánh giá; α chọn trên phần TUNE;
mọi ảnh Visual Genome thuộc VAL (RelTR chưa huấn luyện trên chúng), **không dùng heldout**; ngưỡng, lưới tham số và quy tắc quyết định đã đăng ký trước
(`v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md`, `v6_compositional_benchmarks/PREREGISTRATION_VI.md`, `XAI_PREREGISTRATION_VI.md`,
`final_pipeline/GNN_PREREGISTRATION_VI.md`, `SCENIR_VERIFICATION_VI.md`, `LEVEL4A_PREREGISTRATION_VI.md`).

**Kaggle:** gắn datasets `dannywu375/visualgenome`, `mathurinache/visual-genome`, `clkmuhammed/microsoft-coco-2017-common-objects-in-context`; bật GPU và Internet
(cài `open_clip`, `sentence-transformers`; tải trọng số CLIP, mã + checkpoint RelTR, JSON của ARO / SugarCrepe / What'sUp).
Chạy thử không cần GPU: đặt `P3_SMOKE=1` (dữ liệu giả — không phải kết quả).

## 1. Thiết lập

In [ ]:
# ============================== 1. THIẾT LẬP: thư viện · chế độ chạy · hằng số đã đăng ký trước · tiện ích ==============================
import os, sys, json, csv, time, re, random, hashlib, subprocess, argparse, warnings, glob
from collections import Counter, defaultdict
import numpy as np
warnings.filterwarnings("ignore")

T0 = time.time()
SMOKE = os.environ.get("P3_SMOKE") == "1"            # chạy thử với dữ liệu giả + mô hình giả (KHÔNG phải kết quả)
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.environ.get("P3_OUT", "/kaggle/working/p3_out" if ON_KAGGLE else "./p3_out")
WORK = os.environ.get("P3_WORK", "/kaggle/working/p3_work" if ON_KAGGLE else "./p3_work")
FIG = f"{OUT}/figures"
for _d in (OUT, WORK, FIG): os.makedirs(_d, exist_ok=True)
_LOG = open(f"{OUT}/log.txt", "w", encoding="utf-8")

# Hằng số khoá trước khi chạy (giống hệt các bản đăng ký trước của v5, v6, v6-XAI)
SEEDS = (0, 1, 2)                     # 3 seed: lần rút caption (TN1) / cách chia TUNE-CHECK (TN2)
K_TOP = 100                           # CLIP top-K trước khi xếp hạng lại
ALPHAS = [round(i / 10, 1) for i in range(11)]          # lưới α, có α = 1 (chỉ CLIP)
W_GRID = sorted((a / 4, b / 4, (4 - a - b) / 4) for a in range(5) for b in range(5 - a))   # trọng số 3 kiểu đường đi, bước 0.25
W_OBJ = (0.0, 0.0, 1.0)
TAU = 0.05                            # độ dốc sigmoid của cạnh hình học
FLOOR = 1e-30                         # sàn log như v3
N_BOOT = 10000
SEED_RAND = 20261116
N_RAND_DEL = 10                       # số lần xoá ngẫu nhiên (đối chứng fidelity)
assert len(W_GRID) == 15 and len(ALPHAS) == 11


def elapsed():
    return f"{time.time() - T0:.0f}s"


def p3_log(*a):
    s = f"[{time.time() - T0:8.1f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True); _LOG.write(s + "\n"); _LOG.flush()


def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    p3_log("$", " ".join(cmd)[:150], "->", r.returncode, (r.stderr or "")[-200:].replace("\n", " "))
    return r.returncode


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()


def stream_array(path, chunk=1 << 22):
    """đọc tuần tự một mảng JSON rất lớn (relationships.json ~743 MB) mà không nạp cả file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        buf = f.read(chunk); pos = 0
        while pos < len(buf) and buf[pos] in " \t\r\n﻿": pos += 1
        assert buf[pos] == "[", "top-level JSON is not an array"
        pos += 1
        while True:
            while True:
                while pos < len(buf) and buf[pos] in " \t\r\n,": pos += 1
                if pos < len(buf): break
                more = f.read(chunk)
                if not more: return
                buf = more; pos = 0
            if buf[pos] == "]": return
            while True:
                try:
                    obj, end = dec.raw_decode(buf, pos); break
                except json.JSONDecodeError:
                    more = f.read(chunk)
                    if not more: raise
                    buf = buf[pos:] + more; pos = 0
            pos = end
            yield obj


def fetch(dst, url=None, gdrive=None):
    """tải file công khai (benchmark JSON, checkpoint RelTR) một lần."""
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        if gdrive:
            try:
                import gdown  # noqa
            except ImportError:
                sh([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            sh([sys.executable, "-m", "gdown", gdrive, "-O", dst])
        else:
            sh(["wget", "-q", "-O", dst, url])
    assert os.path.exists(dst) and os.path.getsize(dst) > 0, f"tải thất bại: {url or gdrive}"
    return dst


p3_log(f"Chế độ {'SMOKE (dữ liệu giả)' if SMOKE else 'REAL'} | out={OUT}")


## 2. Dữ liệu nhúng (sinh tự động bởi `build_notebook.py`)
Danh sách id cho luật chống rò rỉ, bảng VG150, số tham chiếu của các lần chạy riêng trước (đối chiếu tái lập).

In [ ]:
# ============================== 2. DỮ LIỆU NHÚNG — nạp từ dataset hieutrinh123/project3-merged-assets ==============================
# Nội dung y hệt cell "2. DỮ LIỆU NHÚNG" gốc (danh sách id chống rò rỉ, bảng, notebook video v5, REF), chuyển ra file
# p3_embedded_data.py vì Kaggle giới hạn mã nguồn notebook 1 MB. Kiểm tra MD5 để chắc nội dung không đổi.
import glob as _g, hashlib as _h
_p = sorted(set(_g.glob("/kaggle/input/*/p3_embedded_data.py") + _g.glob("/kaggle/input/*/*/p3_embedded_data.py")
                + _g.glob("/kaggle/input/*/*/*/p3_embedded_data.py") + _g.glob("./p3_embedded_data.py")))
assert _p, "Thiếu dataset hieutrinh123/project3-merged-assets — Add Input dataset này"
_src = open(_p[0], encoding="utf-8").read()
assert _h.md5(_src.encode()).hexdigest() == "9aacd7a4b43507e253dbf50b72cfe66c", "p3_embedded_data.py khác bản gốc"
exec(compile(_src, _p[0], "exec"))


## 3. Lõi đồ thị
Scene graph là đồ thị có hướng: đỉnh = đối tượng (lớp VG150), cạnh = vị từ (và cạnh hình học suy từ box). Điểm đồ thị của một ứng viên là độ khớp của các **đường đi** từ truy vấn tới ảnh; giải thích chính là các cạnh / đỉnh đã khớp, nên giải thích đến từ cơ chế xếp hạng.

In [ ]:
# ============================== 3. LÕI ĐỒ THỊ: chuẩn hoá · scene graph · tách caption · điểm đồ thị · kết hợp · thống kê ==============================
# Hàm thuần (không đọc file, không mô hình). Gộp nguyên văn logic của g0b_core (phần dùng chung), v5_core và v6_core; chỉ đổi tên hàm trùng.

# ---- 3.1 Bảng VG150 + quan hệ vị từ (tương đương / kéo theo) ----
CLS_IDX = {c: i for i, c in enumerate(TABLES["VG_CLASSES"]) if i > 0}        # chỉ số = cột xác suất lớp của RelTR (cột 0 = 'N/A')
PRED_IDX = {p: i for i, p in enumerate(TABLES["VG_PREDS"]) if i > 0}         # chỉ số = cột xác suất vị từ của RelTR
CANON = {"wears": "wearing", "laying on": "lying on", "watching": "looking at"}   # vị từ tương đương
ENT_RAW = {"on": ["sitting on", "standing on", "lying on", "laying on", "walking on", "parked on", "mounted on", "riding"],
           "in": ["walking in"], "holding": ["carrying"]}                        # vị từ kéo theo: ảnh có p' ⇒ thoả truy vấn p
canon = lambda p: CANON.get(p, p)
ENT = {canon(p): sorted({canon(q) for q in L}) for p, L in ENT_RAW.items()}
FAM = {p: {p} | set(ENT.get(p, [])) for p in {canon(p) for p in PRED_IDX}}


def norm_name(n):
    """tên đối tượng → lớp VG150 (số nhiều đơn giản: shoes → shoe), như v3."""
    n = " ".join(str(n).lower().strip().split())
    if n in CLS_IDX: return n
    for cut in (1, 2):
        if n.endswith("s") and n[:-cut] in CLS_IDX: return n[:-cut]
    return None


def norm_pred(p):
    p = " ".join(str(p).lower().strip().split())
    return canon(p) if p in PRED_IDX else None


def nname(d):
    if d.get("name") is not None: return norm_name(d["name"])
    for n in d.get("names") or []:
        r = norm_name(n)
        if r: return r
    return None


# ---- 3.2 Split (cố định bằng sha1, tái lập được) ----
def dev_g0_split(i):          # pool G0: DEV (35%) / IV khoá (65%)
    return "DEV" if int(hashlib.sha1(("p3-dir2:" + str(i)).encode()).hexdigest(), 16) % 1000 < 350 else "IV"


def p5_split(i):              # Thí nghiệm 1: chia DEV-G0 làm đôi
    return "P-TUNE" if int(hashlib.sha1(("p5-pilot:" + str(i)).encode()).hexdigest(), 16) % 2 == 0 else "P-CHECK"


def v6_split(key, s):         # Thí nghiệm 2: chia theo ảnh, 3 seed
    return "TUNE" if int(hashlib.sha1(f"v6-{s}:{key}".encode()).hexdigest(), 16) % 2 == 0 else "CHECK"


class IdGuard:
    """Hàng rào rò rỉ: chỉ id thuộc tập cho phép mới được đọc / nạp ảnh."""

    def __init__(self, allowed, name):
        self.allowed, self.name, self.n_checked = {int(i) for i in allowed}, name, 0

    def check(self, ids, purpose=""):
        ids = [int(i) for i in ids]
        bad = [i for i in ids if i not in self.allowed]
        if bad: raise PermissionError(f"GUARD[{self.name}]: {len(bad)} id ngoài tập cho phép ({purpose}): {bad[:5]}")
        self.n_checked += len(ids)


# ---- 3.3 Scene graph GT của một ảnh (đỉnh = lớp, cạnh = bộ ba VG150) ----
class Img:
    __slots__ = ("id", "triples", "between", "classes")

    def __init__(s, i):
        s.id = i; s.triples = set(); s.between = defaultdict(list); s.classes = set()


def build_image(i, rel_entries, obj_entries, stats):
    """Dựng scene graph từ annotation VG: bỏ quan hệ trùng, self-loop, thiếu object_id; chuẩn hoá về VG150."""
    im = Img(i)
    for e in obj_entries:
        for o in e.get("objects", []):
            c = nname(o)
            if c: im.classes.add(c)
    seen_rid, seen_key = set(), set()
    for rec in rel_entries:
        for r in rec.get("relationships", []):
            stats["rel_total"] += 1
            rid = r.get("relationship_id")
            if rid in seen_rid: stats["rel_dup_relationship_id"] += 1; continue
            seen_rid.add(rid)
            su, ob = r.get("subject") or {}, r.get("object") or {}
            sid, oid = su.get("object_id", su.get("id")), ob.get("object_id", ob.get("id"))
            if sid is None or oid is None: stats["rel_missing_object_id"] += 1; continue
            if int(sid) == int(oid): stats["rel_self_loop"] += 1; continue
            p_raw = " ".join(str(r.get("predicate", "")).lower().split())
            key = (int(sid), p_raw, int(oid))
            if key in seen_key: stats["rel_duplicate_same_triple_ids"] += 1; continue
            seen_key.add(key)
            s150, o150 = nname(su), nname(ob)
            if s150 and o150:
                im.classes.add(s150); im.classes.add(o150)
                im.between[(s150, o150)].append(p_raw)
                p150 = norm_pred(p_raw)
                if p150: im.triples.add((s150, p150, o150)); stats["edges_vg150"] += 1
    return im


# ---- 3.4 Tách caption thành thực thể + bộ ba (luật, tất định) ----
PRED_PHRASES = {"riding": "riding", "rides": "riding", "ride": "riding", "holding": "holding", "holds": "holding",
                "sitting on": "sitting on", "sits on": "sitting on", "standing on": "standing on", "stands on": "standing on",
                "wearing": "wearing", "wears": "wearing", "next to": "near", "beside": "near", "near": "near",
                "on top of": "on", "on": "on", "in": "in", "inside": "in", "in front of": "in front of", "behind": "behind",
                "under": "under", "underneath": "under", "with": "with", "has": "has", "have": "has", "eating": "eating", "eats": "eating",
                "playing": "playing", "plays": "playing", "looking at": "looking at", "looks at": "looking at",
                "carrying": "carrying", "carries": "carrying", "walking on": "walking on", "walking in": "walking in",
                "laying on": "lying on", "lying on": "lying on", "parked on": "parked on", "hanging from": "hanging from",
                "flying in": "flying in", "covered in": "covered in", "above": "above", "over": "above"}


def _spans(tokens, lookup, maxn=3):
    out, i = [], 0
    while i < len(tokens):
        hit = None
        for n in range(min(maxn, len(tokens) - i), 0, -1):
            v = lookup(" ".join(tokens[i:i + n]))
            if v is not None: hit = (i, i + n, v); break
        if hit: out.append(hit); i = hit[1]
        else: i += 1
    return out


def parse_caption(cap):
    """→ (E: tập lớp, T: tập bộ ba). subject = thực thể gần nhất bên trái vị từ, object = gần nhất bên phải."""
    toks = re.findall(r"[a-z0-9]+", str(cap).lower())
    ents, preds = _spans(toks, norm_name), _spans(toks, PRED_PHRASES.get)
    T = set()
    for (a, b, p) in preds:
        left, right = [e for e in ents if e[1] <= a], [e for e in ents if e[0] >= b]
        if left and right:
            T.add((max(left, key=lambda e: e[1])[2], p, min(right, key=lambda e: e[0])[2]))
    return frozenset(e[2] for e in ents), frozenset(T)


# ---- 3.5 Điểm đồ thị trên scene graph GT (Thí nghiệm 1): ba kiểu đường đi truy vấn → ảnh ----
def match_detail(Tq, Eq, im):
    """s_rel: bộ ba khớp cạnh (cho phép kéo theo) · s_pair: cặp lớp nối bởi cạnh bất kỳ · s_obj: lớp có mặt. Kèm bằng chứng (giải thích)."""
    edges, pairs = [], []
    for (s, p, o) in sorted(Tq):
        for q in sorted(FAM.get(p, {p})):
            if (s, q, o) in im.triples: edges.append(f"{s}|{p}|{o}<={s}|{q}|{o}"); break
        if (s, o) in im.between or (o, s) in im.between: pairs.append(f"{s}|{o}")
    nodes = sorted(c for c in Eq if c in im.classes)
    nt = len(Tq)
    return dict(s_rel=len(edges) / nt if nt else 0.0, s_pair=len(pairs) / nt if nt else 0.0,
                s_obj=len(nodes) / len(Eq) if Eq else 0.0, edges=edges, pairs=pairs, nodes=nodes)


def candidate_components(parsed, topk, graphs):
    Q, K = topk.shape
    out = np.zeros((Q, K, 3))
    for qi, (Eq, Tq) in enumerate(parsed):
        if not Eq and not Tq: continue
        for kj in range(K):
            d = match_detail(Tq, Eq, graphs[topk[qi, kj]])
            out[qi, kj] = (d["s_rel"], d["s_pair"], d["s_obj"])
    return out


def rand_permutation(n, seed=SEED_RAND):
    """hoán vị cố định không có điểm bất động (đối chứng 'đồ thị của ảnh khác')."""
    order = list(range(n)); random.Random(seed).shuffle(order)
    perm = [0] * n
    for pos, j in enumerate(order): perm[j] = order[(pos + 1) % n]
    return perm


def shuffle_predicates(im):
    e = sorted(im.triples); preds = [p for (_, p, _) in e]
    random.Random(int(im.id)).shuffle(preds)
    new = Img(im.id); new.triples = {(s, p, o) for (s, _, o), p in zip(e, preds)}; new.between = im.between; new.classes = im.classes
    return new


# ---- 3.6 Kết hợp CLIP + đồ thị và xếp hạng (Thí nghiệm 1) ----
def zrows(x):
    mu, sd = x.mean(1, keepdims=True), x.std(1, keepdims=True)
    return np.where(sd > 0, (x - mu) / np.where(sd > 0, sd, 1), 0.0)


def fuse_scores(zc, comps, w, alpha):
    g = comps @ np.asarray(w, dtype=np.float64)
    flat = g.max(1) == g.min(1)                  # điểm đồ thị bằng nhau ⇒ giữ thứ tự CLIP
    zg = zrows(g); zg[flat] = 0.0
    return alpha * zc + (1 - alpha) * zg


def rank_of_relevant(scores, rel_pos, clip_rank):
    Q, K = scores.shape
    pos = np.arange(K)[None, :]; rp = np.where(rel_pos >= 0, rel_pos, 0)
    sr = scores[np.arange(Q), rp][:, None]
    better = (scores > sr).sum(1) + ((scores == sr) & (pos < rp[:, None])).sum(1)
    return np.where(rel_pos >= 0, 1 + better, clip_rank).astype(np.int64)


def select_params(grid_mrr):
    return min(grid_mrr, key=lambda k: (-round(grid_mrr[k], 12), -k[1], k[0]))


def tune_grid(zc, comps, rel_pos, clip_rank, w_list, alphas):
    return {(tuple(w), a): float(np.mean(1.0 / rank_of_relevant(fuse_scores(zc, comps, w, a), rel_pos, clip_rank)))
            for w in w_list for a in alphas}


def clip_topk(cos, k):
    Q, N = cos.shape; K = min(k, N)
    order = np.argsort(-cos, axis=1, kind="stable"); topk = order[:, :K]
    pos_rel = np.array([int(np.where(order[i] == i)[0][0]) for i in range(Q)])
    return topk, np.take_along_axis(cos, topk, 1), np.where(pos_rel < K, pos_rel, -1), pos_rel + 1


def metrics_from_ranks(r):
    r = np.asarray(r, dtype=np.float64)
    return dict(R1=float((r <= 1).mean()), R5=float((r <= 5).mean()), R10=float((r <= 10).mean()), MRR=float((1 / r).mean()),
                nDCG10=float(np.where(r <= 10, 1 / np.log2(r + 1), 0.0).mean()))


def bootstrap_delta(d_by_name, n_boot=N_BOOT, seed=SEED_RAND, chunk=1000):
    names = list(d_by_name); n = len(d_by_name[names[0]])
    rng = np.random.default_rng(seed); means = {k: [] for k in names}; done = 0
    while done < n_boot:
        b = min(chunk, n_boot - done); idx = rng.integers(0, n, size=(b, n))
        for k in names: means[k].append(np.asarray(d_by_name[k])[idx].mean(1))
        done += b
    return {k: dict(mean=float(np.mean(d_by_name[k])), lo=float(np.percentile(np.concatenate(means[k]), 2.5)),
                    hi=float(np.percentile(np.concatenate(means[k]), 97.5))) for k in names}


def mean_std(xs):
    a = np.asarray(xs, dtype=np.float64)
    return dict(mean=float(a.mean()), std=float(a.std(ddof=1)) if len(a) > 1 else 0.0, per_seed=[float(x) for x in a])


# ---- 3.7 Điểm đồ thị trên scene graph RelTR (Thí nghiệm 2, 3): cạnh vị từ + cạnh hình học ----
ARO_RX = re.compile(r"^the (.+?) is (.+) the (.+)$")
GEO = {"to the left of": "L", "to the right of": "R"}
ARO_PRED = {"on": "on", "on top of": "on", "wearing": "wearing", "in": "in", "inside": "in", "behind": "behind",
            "in front of": "in front of", "near": "near", "next to": "near", "beside": "near", "by": "near", "of": "of",
            "sitting on": "sitting on", "above": "above", "below": "under", "under": "under", "underneath": "under",
            "holding": "holding", "at": "at", "with": "with", "riding": "riding", "standing on": "standing on", "eating": "eating",
            "walking on": "walking on", "parked on": "parked on", "watching": "looking at", "looking at": "looking at",
            "lying on": "lying on", "walking in": "walking in", "covering": "covering"}


def aro_parse(cap):
    m = ARO_RX.match(cap)
    if not m: return (None, "other", None, None)
    X, r, Y = norm_name(m.group(1)), m.group(2), norm_name(m.group(3))
    if not X or not Y: return (None, "other", r, None)
    if r in GEO: return (X, "geo", GEO[r], Y)
    if r in ARO_PRED: return (X, "pred", ARO_PRED[r], Y)
    return (None, "other", r, None)


def aro_subset(rel_name):
    return "geo" if rel_name in GEO else ("pred" if rel_name in ARO_PRED else "other")


def rel_score(F, X, p, Y, pr=None):
    """cạnh vị từ: log max_j P(sub=X) · P(pred=p) · P(obj=Y) trên 200 truy vấn bộ ba của RelTR (điểm 'rel' của v3)."""
    pr = F["pr"] if pr is None else pr
    v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * pr[:, PRED_IDX[p]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
    j = int(np.argmax(v))
    return float(np.log(max(v[j], FLOOR))), dict(kind="pred", j=j, p=float(v[j]), sub_box=[float(x) for x in F["sb"][j]],
                                                 obj_box=[float(x) for x in F["ob"][j]])


def geo_score(F, X, Y, d, nogeo=False):
    """cạnh hình học: max_{u≠v} P(u=X) · P(v=Y) · σ((cx_v − cx_u)/τ) trên 400 hộp đối tượng (subject + object của 200 truy vấn)."""
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, cx = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0]
    if nogeo: s = np.ones((len(cx), len(cx)))
    else:
        diff = (cx[None, :] - cx[:, None]) if d == "L" else (cx[:, None] - cx[None, :])
        s = 1.0 / (1.0 + np.exp(-diff / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]),
                                                    x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


def aro_graph_score(F, parsed, pr=None, nogeo=False):
    X, kind, r, Y = parsed
    if kind == "geo": return geo_score(F, X, Y, r, nogeo=nogeo)
    if kind == "pred": return rel_score(F, X, r, Y, pr=pr)
    return None, None


def triples_graph_score(F, T, pr=None):
    if not T: return None, None
    vals, ev = [], []
    for (s, p, o) in sorted(T):
        if p not in PRED_IDX: return None, None
        v, e = rel_score(F, s, p, o, pr=pr); vals.append(v); ev.append(dict(triple=f"{s}|{p}|{o}", **e))
    return float(np.mean(vals)), ev


def shuffled_pr(F, key):
    perm = np.arange(F["pr"].shape[1])
    perm[1:] = 1 + np.random.RandomState(int(key) % (2 ** 31)).permutation(F["pr"].shape[1] - 1)
    return F["pr"][:, perm]


def correct(delta):
    d = np.asarray(delta, dtype=np.float64)
    return np.where(d > 0, 1.0, np.where(d == 0, 0.5, 0.0))


def fuse2(dc, dg, valid, alpha, sc, sg):
    dc, dg = np.asarray(dc, float), np.asarray(dg, float)
    g = np.where(valid, dg, 0.0) / sg if sg > 0 else np.zeros_like(dg)
    return np.where(valid, alpha * dc / sc + (1 - alpha) * g, dc / sc)


def tune_alpha(dc, dg, valid):
    sc = float(np.std(dc)) or 1.0
    sg = (float(np.std(np.asarray(dg)[valid])) if np.any(valid) else 1.0) or 1.0
    acc = {a: float(correct(fuse2(dc, dg, valid, a, sc, sg)).mean()) for a in ALPHAS}
    return min(acc, key=lambda a: (-round(acc[a], 12), -a)), sc, sg, acc


def bootstrap_diff_by_cluster(x, y, clusters, n_boot=N_BOOT, seed=SEED_RAND):
    x, y = np.asarray(x, float), np.asarray(y, float)
    keys, inv = np.unique(np.asarray(clusters).astype(str), return_inverse=True); K = len(keys)
    sx, sy, cnt = np.bincount(inv, x, K), np.bincount(inv, y, K), np.bincount(inv, minlength=K).astype(float)
    rng = np.random.default_rng(seed); out = np.empty(n_boot)
    for b0 in range(0, n_boot, 1000):
        b = min(1000, n_boot - b0); idx = rng.integers(0, K, size=(b, K))
        out[b0:b0 + b] = (sx[idx].sum(1) - sy[idx].sum(1)) / cnt[idx].sum(1)
    return dict(mean=float(x.mean() - y.mean()), lo=float(np.percentile(out, 2.5)), hi=float(np.percentile(out, 97.5)))


print(f"Lõi đồ thị sẵn sàng: {len(CLS_IDX)} lớp, {len(PRED_IDX)} vị từ VG150 | {elapsed()}")


## 4. Dữ liệu và luật chống rò rỉ

In [ ]:
# ============================== 4. DỮ LIỆU: tìm input · tải benchmark · lọc chống rò rỉ · scene graph GT · caption COCO ==============================
# Luật rò rỉ (khoá trước):
#  - Mọi ảnh Visual Genome dùng tới phải thuộc VAL của v3 (chỉ số lọc ≥ 83000 ⇒ RelTR chưa huấn luyện trên chúng; heldout KHÔNG dùng).
#  - TN1 chỉ dùng DEV-G0: VAL trừ ảnh benchmark v3, cache v4 và 99 ảnh Phase 5 (danh sách G0, kiểm MD5).
#  - SugarCrepe: ảnh COCO không thuộc VG, hoặc thuộc VG và nằm trong VAL.
assert hashlib.md5(",".join(map(str, VAL_IDS)).encode()).hexdigest() == VAL_MD5 and len(VAL_IDS) == 15128
assert hashlib.md5(",".join(map(str, G0_IDS)).encode()).hexdigest() == G0_MD5 and len(G0_IDS) == 13586
VALSET = set(VAL_IDS)
DEV_G0 = sorted(i for i in G0_IDS if dev_g0_split(i) == "DEV")
assert len(DEV_G0) == 4752 and set(DEV_G0) <= VALSET
GUARD_VAL = IdGuard(VAL_IDS, "VAL")
GUARD_DEV = IdGuard(DEV_G0, "DEV-G0")
p3_log(f"VAL {len(VAL_IDS)} ảnh | DEV-G0 {len(DEV_G0)} ảnh (TN1)")


def find_inputs():
    if os.environ.get("P3_PATHS"): return json.loads(os.environ["P3_PATHS"])
    found, vgdirs, cocoval = defaultdict(list), [], []
    for dp, dn, fn in os.walk("/kaggle/input", followlinks=True):
        b = os.path.basename(dp)
        if b.startswith("VG_100K"):
            if "dannywu375" in dp or "/input/visualgenome/" in dp: vgdirs.append(dp)
            dn[:] = []; continue
        if b == "val2017" and any(f.endswith(".jpg") for f in fn[:50]):
            cocoval.append(dp); dn[:] = []; continue
        if b in ("train2017", "test2017", "unlabeled2017"):
            dn[:] = []; continue
        for f in fn:
            if f in ("image_data.json", "relationships.json", "objects.json", "captions_train2017.json", "captions_val2017.json"):
                found[f].append(os.path.join(dp, f))
    # merged: Kaggle có thể gắn dataset ở /kaggle/input/<owner>/<slug>/ hoặc /kaggle/input/<slug>/ (không có tên chủ sở hữu)
    pick = lambda name, slug: sorted(p for p in found[name] if f"/{slug}/" in p or f"/input/{slug.split('/')[-1]}/" in p)
    r = dict(image_data=pick("image_data.json", "dannywu375/visualgenome"), rel=pick("relationships.json", "mathurinache/visual-genome"),
             obj_dw=pick("objects.json", "dannywu375/visualgenome"), cap_train=sorted(found["captions_train2017.json"]),
             cap_val=sorted(found["captions_val2017.json"]))
    for k, v in r.items(): assert v, f"thiếu input {k} — đã thấy: { {kk: vv[:3] for kk, vv in found.items()} }"
    r = {k: v[0] for k, v in r.items()}
    r.update(vg_dirs=sorted(vgdirs), coco_val=sorted(cocoval)[0])
    return r


P = find_inputs()
SHA = {k: sha256(P[k]) for k in ("image_data", "rel", "obj_dw")}
if not SMOKE:
    for k, v in G0_SHA.items(): assert SHA[k] == v, f"input {k} khác manifest G0"
p3_log("Input VG khớp SHA-256 của G0" if not SMOKE else "SMOKE: bỏ kiểm SHA", "|", {k: P[k] for k in ("image_data", "rel", "obj_dw")})

# ---- benchmark công khai (tải lúc chạy) ----
if SMOKE:
    BENCH = json.loads(os.environ["P3_BENCH"])
else:
    BENCH = dict(aro=fetch(f"{WORK}/visual_genome_relation.json", gdrive=ARO_GDRIVE),
                 replace_rel=fetch(f"{WORK}/replace_rel.json", url=SC_URL + "replace_rel.json"),
                 swap_obj=fetch(f"{WORK}/swap_obj.json", url=SC_URL + "swap_obj.json"))
BSHA = {k: sha256(v) for k, v in BENCH.items()}
p3_log("Benchmark:", BSHA)

# ---- image_data: ánh xạ COCO ↔ VG; chỉ mục file ảnh VG thuộc VAL ----
COCO_OF_VG, VG_OF_COCO = {}, {}
for e in json.load(open(P["image_data"], encoding="utf-8")):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if e.get("coco_id") is not None:
        VG_OF_COCO[int(e["coco_id"])] = i
        if i in VALSET: COCO_OF_VG[i] = int(e["coco_id"])
VG_JPG = {}
for d in P["vg_dirs"]:
    for f in os.listdir(d):
        if f.endswith(".jpg") and f[:-4].isdigit() and int(f[:-4]) in VALSET: VG_JPG.setdefault(int(f[:-4]), os.path.join(d, f))
p3_log(f"Ảnh VG thuộc VAL có file: {len(VG_JPG)}")

# ---- mục benchmark, lọc rò rỉ TRƯỚC khi mở ảnh ----
ITEMS2, EXCL2 = [], Counter()
for k, x in enumerate(json.load(open(BENCH["aro"], encoding="utf-8"))):
    vid = int(x["image_id"])
    if vid not in VALSET: EXCL2["aro_not_VAL"] += 1; continue
    if vid not in VG_JPG: EXCL2["aro_no_jpg"] += 1; continue
    box = (int(x["bbox_x"]), int(x["bbox_y"]), int(x["bbox_x"]) + int(x["bbox_w"]), int(x["bbox_y"]) + int(x["bbox_h"]))
    ITEMS2.append(dict(bench="aro", item=f"aro{k}", key=str(vid), vg_id=vid, path=VG_JPG[vid], crop=box, true=x["true_caption"],
                       false=x["false_caption"], subset=aro_subset(x["relation_name"]), rel=x["relation_name"],
                       xid=int(x["primary_object_id"]), yid=int(x["relation_info"]["object"])))
for b in ("replace_rel", "swap_obj"):
    for k, x in json.load(open(BENCH[b], encoding="utf-8")).items():
        coco = int(x["filename"].split(".")[0]); vid = VG_OF_COCO.get(coco)
        if vid is not None and vid not in VALSET: EXCL2[f"{b}_vg_not_VAL"] += 1; continue
        path = os.path.join(P["coco_val"], x["filename"])
        if not os.path.exists(path): EXCL2[f"{b}_no_jpg"] += 1; continue
        ITEMS2.append(dict(bench=b, item=f"{b}{k}", key=f"coco{coco}", vg_id=vid, path=path, crop=None, true=x["caption"],
                           false=x["negative_caption"], subset="vg_VAL" if vid else "not_in_VG", rel="", xid=None, yid=None))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2 if it["vg_id"] is not None], "benchmark items")
p3_log("Mục benchmark:", dict(Counter(i["bench"] for i in ITEMS2)), "| loại:", dict(EXCL2))

# ---- annotation VG: chỉ cho ảnh cần (DEV-G0 cho TN1; ảnh ARO cho box GT của XAI) ----
NEED_REL = set(DEV_G0)
NEED_OBJ = set(DEV_G0) | {it["vg_id"] for it in ITEMS2 if it["bench"] == "aro"}
GUARD_VAL.check(NEED_OBJ, "annotation")
REL, OBJ = defaultdict(list), defaultdict(list)
for e in stream_array(P["rel"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_REL: REL[i].append(e)                 # mọi ảnh khác bị bỏ ngay, không lưu, không đếm
for e in stream_array(P["obj_dw"]):
    i = int(e["image_id"] if "image_id" in e else e["id"])
    if i in NEED_OBJ: OBJ[i].append(e)
GTBOX = {int(o["object_id"]): (int(o["x"]), int(o["y"]), int(o["x"]) + int(o["w"]), int(o["y"]) + int(o["h"]))
         for i in NEED_OBJ for e in OBJ.get(i, []) for o in e.get("objects", [])}
p3_log(f"Annotation: relationships {len(REL)} ảnh, objects {len(OBJ)} ảnh, {len(GTBOX)} hộp GT | {elapsed()}")

# ---- caption COCO cho TN1 ----
CAPS = defaultdict(list)
for k in ("cap_train", "cap_val"):
    for a in json.load(open(P[k], encoding="utf-8"))["annotations"]:
        CAPS[int(a["image_id"])].append((int(a["id"]), " ".join(str(a["caption"]).split())))
for v in CAPS.values(): v.sort()
EXCL1 = Counter()
ELIG1 = []
for i in DEV_G0:
    if i not in COCO_OF_VG: continue
    if len(CAPS.get(COCO_OF_VG[i], [])) < 3: EXCL1["lt3_captions"] += 1; continue
    if i not in VG_JPG: EXCL1["no_jpg"] += 1; continue
    ELIG1.append(i)
GUARD_DEV.check(ELIG1, "TN1 ảnh")
p3_log(f"TN1: {len(ELIG1)} ảnh DEV-G0 có caption COCO | loại {dict(EXCL1)} | {elapsed()}")


## 4v. Cấp 4 (b) — Mở rộng video MSVD (text-to-video)
Chạy nguyên vẹn notebook video Level4B v5 "EvidenceGraph" (đã chạy trên Kaggle, mã trùng khớp) trong tiến trình riêng, **trước** phần ảnh để GPU còn trống cho Qwen2.5-VL. Kết quả được đưa vào bảng tổng hợp ở mục 11. Khoảng 2–2,5 giờ.

## 4u. Phần C — Level 4(b) v2-graphfix (text → video, MSVD + MSR-VTT)
Notebook `lamvo30ai/project03-level4b-v2-graphfix` (version 1) được **nhúng nguyên văn** và chạy trong tiến trình riêng (papermill), giống cách mục 4v chạy notebook video v5, để không đụng biến của notebook này. Nó dựng đồ thị v3 có vị từ, TypedSAGE / QC-RGAT, tagger học từ caption train, cạnh ConceptNet, xAI theo thứ hạng; và tạo **cache đặc trưng frame CLIP** của MSVD / MSR-VTT mà notebook video v5 (mục 4v) dùng lại.

Chế độ chạy (`P3_RUN_L4B_V2`): `auto` (mặc định) — nếu output của notebook này đã được Add Input (thấy cache `frames_msvd_*.npz`) thì **không chạy lại**, chỉ chép kết quả; nếu không thấy thì chạy đầy đủ (~2,4 giờ trên T4). `1` = luôn chạy, `0` = không chạy.


In [ ]:
# ============================== 4u. PHẦN C — Level 4(b) v2-graphfix: nhúng nguyên văn, chạy trong tiến trình riêng ==============================
# Phải chạy TRƯỚC mục 4v: notebook video v5 dùng lại cache đặc trưng frame mà notebook này tạo ra (qua MSVD_L4_CACHE_DIRS).
_p2 = sorted(set(glob.glob("/kaggle/input/*/project03-level4b-v2-graphfix.ipynb") + glob.glob("/kaggle/input/*/*/project03-level4b-v2-graphfix.ipynb")
                 + glob.glob("/kaggle/input/*/*/*/project03-level4b-v2-graphfix.ipynb")))
L4B_V2_NB_JSON = open(_p2[0], encoding="utf-8").read() if _p2 else ""   # notebook gốc, nạp từ dataset hieutrinh123/project3-merged-assets (giới hạn 1 MB)
L4B_V2_NB_MD5 = 'f4b127817568994b192bbd399f9fc44b'
L4B_V2_DIR = "/kaggle/working/project03_level4b_v2" if ON_KAGGLE else f"{WORK}/project03_level4b_v2"
L4B_V2_RES = f"{OUT}/video_v2"
os.makedirs(L4B_V2_RES, exist_ok=True)
RES_L4B_V2 = dict(status="not run")
_t_v2 = time.time()


def _copy_v2_results(src_dir):
    n = 0
    for f in glob.glob(f"{src_dir}/*.csv") + glob.glob(f"{src_dir}/*.json") + glob.glob(f"{src_dir}/*.md"):
        try:
            open(f"{L4B_V2_RES}/{os.path.basename(f)}", "wb").write(open(f, "rb").read()); n += 1
        except Exception:
            pass
    return n


try:
    _mode = os.environ.get("P3_RUN_L4B_V2", "auto")
    # chỉ dò vài mức thư mục cố định (không rglob cả /kaggle/input — COCO/VG có hàng trăm nghìn file)
    _cached = sorted(set(glob.glob("/kaggle/input/*/project03_level4b_v2/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/cache/frames_msvd_*.npz") +
                         glob.glob("/kaggle/input/*/*/project03_level4b_v2/cache/frames_msvd_*.npz"))) if ON_KAGGLE else []
    if SMOKE:
        RES_L4B_V2 = dict(status="skipped (SMOKE)")
    elif _mode == "0" or (_mode == "auto" and _cached):
        _root = os.path.dirname(os.path.dirname(_cached[0])) if _cached else None
        _n = _copy_v2_results(f"{_root}/results") if _root else 0
        RES_L4B_V2 = dict(status="reused" if _cached else "skipped", cache=_cached[:3], results_copied=_n, source=_root)
        p3_log("Phần C: không chạy lại —", ("dùng output đã Add Input: " + _root) if _root else "P3_RUN_L4B_V2=0", f"| chép {_n} file kết quả")
    else:
        _nb_in = f"{WORK}/level4b_v2_graphfix_input.ipynb"
        _nb_out = f"{L4B_V2_RES}/project03_level4b_v2_graphfix_executed.ipynb"
        assert hashlib.md5(L4B_V2_NB_JSON.encode()).hexdigest() == L4B_V2_NB_MD5, "notebook v2-graphfix nhúng bị thay đổi"
        open(_nb_in, "w", encoding="utf-8").write(L4B_V2_NB_JSON)
        try:
            import papermill as pm
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
        _old_out = os.environ.get("MSVD_L4_OUT")
        os.environ["MSVD_L4_OUT"] = L4B_V2_DIR          # chỉ cho tiến trình v2; khôi phục ngay sau đó để 4v (v5) dùng thư mục của nó
        try:
            p3_log("Phần C: bắt đầu chạy notebook Level 4(b) v2-graphfix (tiến trình riêng)…")
            pm.execute_notebook(_nb_in, _nb_out, cwd="/kaggle/working" if ON_KAGGLE else WORK,
                                kernel_name="python3", log_output=True, progress_bar=False)
        finally:
            if _old_out is None:
                os.environ.pop("MSVD_L4_OUT", None)
            else:
                os.environ["MSVD_L4_OUT"] = _old_out
        _n = _copy_v2_results(f"{L4B_V2_DIR}/results")
        RES_L4B_V2 = dict(status="ok", out_dir=L4B_V2_DIR, results_copied=_n)
    # cache của Phần C → notebook video v5 (mục 4v) tìm thấy qua MSVD_L4_CACHE_DIRS
    _cdirs = [d for d in os.environ.get("MSVD_L4_CACHE_DIRS", "").split(":") if d]
    for _d in ([f"{L4B_V2_DIR}/cache"] + [os.path.dirname(p) for p in _cached[:1]]):
        if os.path.isdir(_d) and _d not in _cdirs:
            _cdirs.append(_d)
    os.environ["MSVD_L4_CACHE_DIRS"] = ":".join(_cdirs)
    RES_L4B_V2["cache_dirs"] = _cdirs
except Exception as ex:
    RES_L4B_V2 = dict(status="error", error=repr(ex)[:1500])
    p3_log("Phần C: LỖI — các phần sau vẫn chạy tiếp:", repr(ex)[:300])
RES_L4B_V2["minutes"] = round((time.time() - _t_v2) / 60, 1)
p3_log("Phần C:", {k: v for k, v in RES_L4B_V2.items() if k != "error"})


In [ ]:
# ============================== 4v. CẤP 4 (b) — MỞ RỘNG VIDEO (text-to-video, MSVD): chạy nguyên vẹn notebook video v5 đã kiểm chứng ==============================
# Notebook video Level4B v5 "EvidenceGraph" (project/code/project03_level4b/video_v5/, đã chạy trên Kaggle — mã trùng khớp) được NHÚNG nguyên văn
# và chạy trong một tiến trình riêng (papermill) để không đụng biến của notebook này. Chạy TRƯỚC phần ảnh để GPU còn trống cho Qwen2.5-VL-7B.
# Ô này KHÔNG import torch (không khởi tạo CUDA ở tiến trình chính). Lỗi ở nhánh video được ghi lại, phần ảnh vẫn chạy tiếp.
VIDEO_DIR = f"{OUT}/video"
os.makedirs(VIDEO_DIR, exist_ok=True)
RES_VIDEO = dict(status="not run")
_t_video = time.time()
try:
    _nb_in = f"{WORK}/video_v5_input.ipynb"
    _nb_out = f"{VIDEO_DIR}/Project03_Level4B_v5_EvidenceGraph_executed.ipynb"
    if SMOKE:   # chạy thử: notebook giả chỉ ghi một summary giả đúng cấu trúc (KHÔNG phải kết quả MSVD)
        _fake = {"cells": [{"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": [
            "import json, os\n", f"d = {VIDEO_DIR!r} + '/fake_run/results'\n", "os.makedirs(d, exist_ok=True)\n",
            "json.dump({'test': [{'model': 'M0_clip', 'set': 'test_all', 'MRR': 0.5, 'R@1': 0.4, 'R@5': 0.7, 'R@10': 0.8, 'n_queries': 10},"
            "{'model': 'M4_full_graph', 'set': 'test_all', 'MRR': 0.6, 'R@1': 0.5, 'R@5': 0.8, 'R@10': 0.9, 'n_queries': 10}],"
            "'comparisons': [], 'fidelity': {}, 'test_looks_msvd': 'smoke'}, open(d + '/v5_summary.json', 'w'))\n"]}],
            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}}, "nbformat": 4, "nbformat_minor": 5}
        json.dump(_fake, open(_nb_in, "w"))
        _cwd = VIDEO_DIR
    else:
        open(_nb_in, "w", encoding="utf-8").write(VIDEO_NB_JSON)
        assert hashlib.md5(VIDEO_NB_JSON.encode()).hexdigest() == VIDEO_NB_MD5, "notebook video nhúng bị thay đổi"
        _cwd = "/kaggle/working" if ON_KAGGLE else VIDEO_DIR
    try:
        import papermill as pm
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "papermill"]); import papermill as pm
    p3_log("Video: bắt đầu chạy notebook video v5 (tiến trình riêng)…")
    pm.execute_notebook(_nb_in, _nb_out, cwd=_cwd, kernel_name="python3", log_output=True, progress_bar=False)
    _hits = sorted(glob.glob(f"{_cwd}/**/results/v5_summary.json", recursive=True), key=os.path.getmtime)
    assert _hits, "không thấy v5_summary.json sau khi chạy notebook video"
    _vs = json.load(open(_hits[-1], encoding="utf-8"))
    _res_dir = os.path.dirname(_hits[-1])
    for f in glob.glob(f"{_res_dir}/*.csv") + glob.glob(f"{_res_dir}/*.json"):
        try:
            open(f"{VIDEO_DIR}/{os.path.basename(f)}", "wb").write(open(f, "rb").read())
        except Exception:
            pass
    RES_VIDEO = dict(status="ok", summary_path=_hits[-1], test=_vs.get("test", []), comparisons=_vs.get("comparisons", []),
                     fidelity=_vs.get("fidelity", {}), test_looks_msvd=_vs.get("test_looks_msvd"), vlm=_vs.get("vlm", {}))
except Exception as ex:
    RES_VIDEO = dict(status="error", error=repr(ex)[:1500])
    p3_log("Video: LỖI — phần ảnh vẫn chạy tiếp:", repr(ex)[:300])
RES_VIDEO["minutes"] = round((time.time() - _t_video) / 60, 1)
if RES_VIDEO["status"] == "ok":
    _tv = {r["model"]: r for r in RES_VIDEO["test"] if r.get("set") == "test_all"}
    p3_log("Video xong:", {k: round(v["MRR"], 4) for k, v in _tv.items()}, f"| {RES_VIDEO['minutes']} phút | lần nhìn test MSVD: {RES_VIDEO['test_looks_msvd']}")


## 5. Mô hình đóng băng (CLIP, RelTR)

In [ ]:
# ============================== 5. MÔ HÌNH ĐÓNG BĂNG: CLIP ViT-L/14 · RelTR (scene graph từ điểm ảnh) — không huấn luyện ==============================
NQ = 200   # số truy vấn bộ ba của RelTR

if SMOKE:
    from PIL import Image   # chỉ để có kiểu; ảnh giả không được mở
    MODEL_INFO = "fake (smoke)"
    _h = lambda s: int(hashlib.md5(str(s).encode()).hexdigest(), 16) % 2 ** 32

    def clip_image_paths(paths):
        v = np.stack([np.random.default_rng(_h(p)).standard_normal(16) for p in paths])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def clip_text(caps):
        v = np.stack([np.random.default_rng(_h(c)).standard_normal(16) for c in caps])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def reltr_clip_features(items, flip=False):
        """giả lập: cài bằng chứng cho caption ĐÚNG của mục ARO (để kiểm tra luồng xử lý, KHÔNG phải kết quả)."""
        F = []
        for it in items:
            r = np.random.default_rng(_h(it["item"]))
            ps, po, pr = r.random((NQ, 151)) * 0.01, r.random((NQ, 151)) * 0.01, r.random((NQ, 51)) * 0.01
            sb, ob = r.random((NQ, 4)) * 0.5 + 0.25, r.random((NQ, 4)) * 0.5 + 0.25
            par = aro_parse(it["true"]) if it["bench"] == "aro" else None
            if par and par[1] != "other":
                X, kind, rel, Y = par
                ps[0, CLS_IDX[X]] = po[0, CLS_IDX[Y]] = 0.9
                if kind == "pred": pr[0, PRED_IDX[rel]] = 0.9
                sb[0, :2], ob[0, :2] = ((0.2, 0.5), (0.8, 0.5)) if rel == "L" else ((0.8, 0.5), (0.2, 0.5))
                sb[0, 2:] = ob[0, 2:] = 0.1
            if flip: sb[:, 0], ob[:, 0] = 1 - sb[:, 0], 1 - ob[:, 0]
            img = r.standard_normal(16)
            F.append(dict(ps=ps.astype(np.float16), po=po.astype(np.float16), pr=pr.astype(np.float16), sb=sb.astype(np.float16),
                          ob=ob.astype(np.float16), img=img / np.linalg.norm(img)))
        return F

    def pil_of(it, flip=False):
        return Image.new("RGB", (100, 100), (128, 128, 128))

    def pil_path(p):
        return Image.new("RGB", (100, 100), (128, 128, 128))
else:
    try:
        import open_clip
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"]); import open_clip
    import torch, torchvision
    import torchvision.transforms as TT
    from PIL import Image, ImageOps
    from concurrent.futures import ThreadPoolExecutor
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    # RelTR: mã nguồn GitHub + checkpoint VG công khai (cùng cách v3)
    RDIR = f"{WORK}/RelTR"
    if not os.path.isdir(f"{RDIR}/models"):
        assert sh(["git", "clone", "-q", "--depth", "1", "https://github.com/yrcong/RelTR.git", RDIR]) == 0
    CK = fetch(f"{WORK}/reltr_vg_checkpoint0149.pth", gdrive=RELTR_GDRIVE)
    CK_SHA = sha256(CK)
    _orig = torchvision.models.resnet50
    torchvision.models.resnet50 = lambda *a, pretrained=False, **kw: _orig(*a, weights=None, **kw)
    sys.path.insert(0, RDIR)
    try:
        from models import build_model as _build
    finally:
        sys.path.remove(RDIR); torchvision.models.resnet50 = _orig
    _args = argparse.Namespace(lr_backbone=1e-5, dataset="vg", backbone="resnet50", dilation=False, position_embedding="sine",
                               enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8,
                               num_entities=100, num_triplets=NQ, pre_norm=False, aux_loss=False, device=DEVICE,
                               set_cost_class=1, set_cost_bbox=5, set_cost_giou=2, set_iou_threshold=0.7, bbox_loss_coef=5,
                               giou_loss_coef=2, rel_loss_coef=1, eos_coef=0.1, return_interm_layers=False)
    RELTR, _, _ = _build(_args)
    _sd = torch.load(CK, map_location="cpu", weights_only=False)
    p3_log("RelTR:", RELTR.load_state_dict(_sd["model"])); del _sd
    RELTR = RELTR.to(DEVICE).eval()
    RTF = TT.Compose([TT.Resize(800, max_size=1333), TT.ToTensor(), TT.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    CLIP, _, PRE = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_WEIGHTS)
    CLIP = CLIP.to(DEVICE).eval(); TOK = open_clip.get_tokenizer(CLIP_MODEL)
    MODEL_INFO = f"RelTR ckpt sha256 {CK_SHA} | CLIP {CLIP_MODEL}/{CLIP_WEIGHTS} | open_clip {open_clip.__version__} | torch {torch.__version__} | {DEVICE}"
    p3_log(MODEL_INFO)

    def pil_path(p):
        return Image.open(p).convert("RGB")

    def pil_of(it, flip=False):
        im = pil_path(it["path"])
        if it["crop"]: im = im.crop(it["crop"])                  # ARO: cắt theo bbox của mục (giao thức ARO)
        return ImageOps.mirror(im) if flip else im

    @torch.no_grad()
    def clip_image_paths(paths, bs=64):
        out = []
        with ThreadPoolExecutor(8) as ex:
            for s in range(0, len(paths), bs):
                x = list(ex.map(lambda p: PRE(pil_path(p)), paths[s:s + bs]))
                with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                    f = CLIP.encode_image(torch.stack(x).to(DEVICE)).float()
                out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def clip_text(caps, bs=256):
        out = []
        for s in range(0, len(caps), bs):
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                f = CLIP.encode_text(TOK(caps[s:s + bs]).to(DEVICE)).float()
            out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def reltr_clip_features(items, flip=False, bs=4):
        """RelTR: softmax rồi bỏ cột nền (cột cuối), 200 truy vấn bộ ba, hộp cxcywh chuẩn hoá — như v3. Kèm đặc trưng CLIP của cùng ảnh."""
        F = []
        for s in range(0, len(items), bs):
            pils = [pil_of(it, flip) for it in items[s:s + bs]]
            try:
                o = RELTR([RTF(im).to(DEVICE) for im in pils])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                parts = [RELTR([RTF(im).to(DEVICE)]) for im in pils]
                o = {k: torch.cat([q[k] for q in parts]) for k in ("sub_logits", "obj_logits", "rel_logits", "sub_boxes", "obj_boxes")}
            sm = lambda k: o[k].float().softmax(-1)[..., :-1].cpu().numpy().astype(np.float16)
            bx = lambda k: o[k].float().cpu().numpy().astype(np.float16)
            ps, po, pr, sb, ob = sm("sub_logits"), sm("obj_logits"), sm("rel_logits"), bx("sub_boxes"), bx("obj_boxes")
            assert ps.shape[1:] == (NQ, 151) and pr.shape[1:] == (NQ, 51)
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                ci = CLIP.encode_image(torch.stack([PRE(im) for im in pils]).to(DEVICE)).float()
            ci = torch.nn.functional.normalize(ci, dim=-1).cpu().numpy()
            for k in range(len(pils)):
                F.append(dict(ps=ps[k], po=po[k], pr=pr[k], sb=sb[k], ob=ob[k], img=ci[k]))
            if (s // bs) % 200 == 0: p3_log(f"  RelTR+CLIP{' (lật)' if flip else ''} {len(F)}/{len(items)}")
        return F


## 6. Thí nghiệm 1 — văn bản → ảnh, full-gallery (bộ chấm đường đi trên scene graph)

In [ ]:
# ============================== 6. THÍ NGHIỆM 1 — Văn bản → ảnh (nhánh A), full-gallery: CLIP top-K → điểm đồ thị → xếp hạng lại ==============================
# Đăng ký trước: v5_vgcoco_pilot/PILOT_PREREGISTRATION_VI.md. Truy vấn = caption COCO (người viết khác VG); đồ thị = scene graph GT của VG.
# Tham số (w, α) chọn trên P-TUNE; P-CHECK chỉ đánh giá. Đối chứng: G-obj (chỉ đối tượng), G-rand (đồ thị ảnh khác), G-shuf (hoán vị vị từ).
GSTATS = Counter()
GRAPH = {i: build_image(i, REL.get(i, []), OBJ.get(i, []), GSTATS) for i in ELIG1}
FEAT1 = dict(zip(ELIG1, clip_image_paths([VG_JPG[i] for i in ELIG1])))
SPL1 = {"P-TUNE": [i for i in ELIG1 if p5_split(i) == "P-TUNE"], "P-CHECK": [i for i in ELIG1 if p5_split(i) == "P-CHECK"]}
CAPTION = {(i, c): CAPS[COCO_OF_VG[i]][c][1] for i in ELIG1 for c in SEEDS}
METHODS1 = ("G-full", "G-obj", "G-rand", "G-shuf")
W_LISTS = {"G-full": W_GRID, "G-obj": [W_OBJ], "G-rand": W_GRID, "G-shuf": W_GRID}
p3_log(f"TN1: P-TUNE {len(SPL1['P-TUNE'])} / P-CHECK {len(SPL1['P-CHECK'])} ảnh | {elapsed()}")


def exp1_split(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    parsed = [parse_caption(x) for x in caps]
    g_true = [GRAPH[i] for i in ids]
    perm = rand_permutation(len(ids))
    comps = {"G-full": candidate_components(parsed, topk, g_true),
             "G-rand": candidate_components(parsed, topk, [g_true[perm[j]] for j in range(len(ids))]),
             "G-shuf": candidate_components(parsed, topk, [shuffle_predicates(g) for g in g_true])}
    comps["G-obj"] = comps["G-full"]
    return dict(ids=ids, caps=caps, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=parsed, comps=comps,
                g_true=g_true, T=T, I=I)


RES1 = {"seeds": {}}
EXP1_KEEP = {}
for c in SEEDS:
    tune, check = exp1_split("P-TUNE", c), exp1_split("P-CHECK", c)
    params, alpha_curve = {}, {}
    for m in METHODS1:
        grid = tune_grid(tune["zc"], tune["comps"][m], tune["rel_pos"], tune["clip_rank"], W_LISTS[m], ALPHAS)
        params[m] = (select_params(grid), grid[select_params(grid)])
        alpha_curve[m] = {a: max(v for (w, aa), v in grid.items() if aa == a) for a in ALPHAS}   # độ nhạy α: MRR tốt nhất theo w, trên TUNE
    ranks, fused = {"CLIP": check["clip_rank"]}, {}
    for m in METHODS1:
        (w, a), _ = params[m]
        fused[m] = fuse_scores(check["zc"], check["comps"][m], w, a)
        ranks[m] = rank_of_relevant(fused[m], check["rel_pos"], check["clip_rank"])
    rr = {k: 1.0 / v for k, v in ranks.items()}
    has_T = np.array([len(T) > 0 for E, T in check["parsed"]])
    RES1["seeds"][c] = dict(
        params={m: dict(w=list(params[m][0][0]), alpha=params[m][0][1], tune_MRR=params[m][1]) for m in METHODS1},
        n_tune=len(tune["ids"]), n_check=len(check["ids"]), check={k: metrics_from_ranks(v) for k, v in ranks.items()},
        candidate_recall_at_K=float((check["clip_rank"] <= K_TOP).mean()), alpha_curve_tune_MRR=alpha_curve,
        parse_coverage=dict(entity=float(np.mean([len(E) > 0 for E, T in check["parsed"]])), triple=float(has_T.mean())),
        bootstrap=bootstrap_delta({"H1 full-CLIP": rr["G-full"] - rr["CLIP"], "H2 full-obj": rr["G-full"] - rr["G-obj"],
                                   "H3 full-rand": rr["G-full"] - rr["G-rand"], "H4 full-shuf": rr["G-full"] - rr["G-shuf"]}),
        subgroups={nm: dict(n=int(mk.sum()), **({k: metrics_from_ranks(v[mk]) for k, v in ranks.items()} if mk.any() else {}))
                   for nm, mk in (("co_bo_ba", has_T), ("khong_bo_ba", ~has_T))})
    if c == 0: EXP1_KEEP = dict(check=check, params=params, fused=fused, ranks=ranks)
    p3_log(f"TN1 seed {c}: MRR", {k: round(v["MRR"], 4) for k, v in RES1["seeds"][c]["check"].items()}, "| H1", RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"])

H1 = {c: RES1["seeds"][c]["bootstrap"]["H1 full-CLIP"] for c in SEEDS}
RES1["decision"] = "GO" if all(H1[c]["lo"] > 0 for c in SEEDS) else "NO-GO"
RES1["mean_std"] = {k: {m: mean_std([RES1["seeds"][c]["check"][k][m] for c in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                    for k in ["CLIP", *METHODS1]}
RES1.update(n_eligible=len(ELIG1), excluded=dict(EXCL1), graph_stats=dict(GSTATS))
p3_log("TN1 quyết định (đăng ký trước):", RES1["decision"], "| MRR trung bình:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1["mean_std"].items()})


## 6b. Thí nghiệm 1b — GraphSAGE (GNN đã học)
Đăng ký trước: `final_pipeline/GNN_PREREGISTRATION_VI.md`. Đồ thị dị thể ảnh – khái niệm; học dự đoán liên kết trên ảnh không thuộc P-CHECK; suy diễn quy nạp một chiều; ablation cạnh ngẫu nhiên và bỏ cạnh.

In [ ]:
# ============================== 6b. THÍ NGHIỆM 1b — GraphSAGE (GNN đã học) làm điểm đồ thị trong cùng pipeline văn bản → ảnh ==============================
# Đăng ký trước: final_pipeline/GNN_PREREGISTRATION_VI.md. Đồ thị dị thể ảnh – khái niệm (lớp VG150, bộ ba VG150) từ scene graph GT của chính ảnh.
# Huấn luyện chỉ trên ảnh DEV-G0 KHÔNG thuộc P-CHECK; suy diễn quy nạp một chiều (ảnh P-CHECK nhận tin, không gửi tin). α chọn trên P-TUNE.
import torch, torch.nn as nn, torch.nn.functional as TF_
GDEV = "cuda" if torch.cuda.is_available() else "cpu"
SAGE_DIM, SAGE_EPOCHS, SAGE_LR, SAGE_WD, SAGE_DROP, SAGE_MSG_FRAC, SAGE_LOGIT_SCALE = 256, 300, 1e-3, 1e-5, 0.1, 0.8, 10.0

EVAL_IMGS = list(SPL1["P-CHECK"]); EVAL_SET = set(EVAL_IMGS)
TRAIN_IMGS = sorted(i for i in DEV_G0 if i in VG_JPG and i not in EVAL_SET)
GUARD_DEV.check(TRAIN_IMGS + EVAL_IMGS, "GraphSAGE")
assert not set(TRAIN_IMGS) & set(EVAL_IMGS)
for i in TRAIN_IMGS:
    if i not in GRAPH: GRAPH[i] = build_image(i, REL.get(i, []), OBJ.get(i, []), Counter())
_miss = [i for i in TRAIN_IMGS if i not in FEAT1]
FEAT1.update(dict(zip(_miss, clip_image_paths([VG_JPG[i] for i in _miss]))) if _miss else {})
IMG_NODES = TRAIN_IMGS + EVAL_IMGS
IMG_IDX = {i: k for k, i in enumerate(IMG_NODES)}
CLS_LIST = sorted(CLS_IDX)
TRI_LIST = sorted({t for i in TRAIN_IMGS for t in GRAPH[i].triples})          # chỉ bộ ba có trong ảnh huấn luyện
CON = {("c", c): k for k, c in enumerate(CLS_LIST)}
CON.update({("t", t): len(CLS_LIST) + k for k, t in enumerate(TRI_LIST)})
NI, NC = len(IMG_NODES), len(CON)
X_IMG = torch.tensor(np.stack([FEAT1[i] for i in IMG_NODES]), dtype=torch.float32)
X_CON = torch.tensor(clip_text([f"a photo of a {c}" for c in CLS_LIST] + [f"a photo of a {s} {p} a {o}" for s, p, o in TRI_LIST]), dtype=torch.float32)
p3_log(f"GraphSAGE: {len(TRAIN_IMGS)} ảnh huấn luyện, {len(EVAL_IMGS)} ảnh P-CHECK, {len(CLS_LIST)} lớp + {len(TRI_LIST)} bộ ba | {elapsed()}")


def image_edges(rand_seed=None):
    """danh sách (ảnh_idx, khái_niệm_idx). rand_seed ≠ None ⇒ cạnh ngẫu nhiên giữ nguyên bậc từng loại (ablation SAGE-rand)."""
    E = []
    rr = np.random.default_rng(rand_seed) if rand_seed is not None else None
    for i in IMG_NODES:
        im = GRAPH[i]
        cl = [CON[("c", c)] for c in sorted(im.classes) if ("c", c) in CON]
        tr = [CON[("t", t)] for t in sorted(im.triples) if ("t", t) in CON]
        if rr is not None:
            cl = list(rr.choice(len(CLS_LIST), size=len(cl), replace=False)) if cl else []
            tr = list(len(CLS_LIST) + rr.choice(len(TRI_LIST), size=min(len(tr), len(TRI_LIST)), replace=False)) if tr else []
        E += [(IMG_IDX[i], int(c)) for c in cl + tr]
    return np.array(E, dtype=np.int64).reshape(-1, 2)


def mean_adj(edges_img_recv, edges_con_recv, n):
    """ma trận thưa chuẩn hoá theo hàng (mean aggregator). Đỉnh khái niệm có chỉ số NI + k."""
    rows = np.concatenate([edges_img_recv[:, 0], NI + edges_con_recv[:, 1]])
    cols = np.concatenate([NI + edges_img_recv[:, 1], edges_con_recv[:, 0]])
    deg = np.bincount(rows, minlength=n).astype(np.float32)
    vals = 1.0 / np.maximum(deg[rows], 1.0)
    return torch.sparse_coo_tensor(torch.tensor(np.stack([rows, cols])), torch.tensor(vals), (n, n)).coalesce()


class SAGE(nn.Module):
    def __init__(self, d_img, d_con, d=SAGE_DIM, layers=2):
        super().__init__()
        self.pi, self.pc = nn.Linear(d_img, d), nn.Linear(d_con, d)
        self.ws = nn.ModuleList(nn.Linear(d, d) for _ in range(layers))
        self.wn = nn.ModuleList(nn.Linear(d, d, bias=False) for _ in range(layers))

    def forward(self, xi, xc, M):
        h = torch.cat([self.pi(xi), self.pc(xc)])
        for l in range(len(self.ws)):
            h = self.ws[l](h) + self.wn[l](torch.sparse.mm(M, h))
            if l < len(self.ws) - 1: h = TF_.dropout(TF_.relu(h), SAGE_DROP, self.training)
        return TF_.normalize(h, dim=-1)


def train_sage(seed, layers=2, rand=False):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    E = image_edges(SEED_RAND + seed if rand else None)
    is_train = np.array([IMG_NODES[a] not in EVAL_SET for a in E[:, 0]]) if len(E) else np.zeros(0, bool)
    Etr = E[is_train]
    xi, xc = X_IMG.to(GDEV), X_CON.to(GDEV)
    model = SAGE(xi.shape[1], xc.shape[1], layers=layers).to(GDEV)
    opt = torch.optim.Adam(model.parameters(), lr=SAGE_LR, weight_decay=SAGE_WD)
    is_cls = Etr[:, 1] < len(CLS_LIST)
    losses = []
    for ep in range(SAGE_EPOCHS):
        model.train()
        perm = rng.permutation(len(Etr)); cut = int(SAGE_MSG_FRAC * len(Etr))
        msg, sup = Etr[perm[:cut]], Etr[perm[cut:]]
        M = mean_adj(msg, msg, NI + NC).to(GDEV)          # chỉ ảnh huấn luyện gửi/nhận tin trong lúc học
        h = model(xi, xc, M)
        neg = np.where(sup[:, 1] < len(CLS_LIST), rng.integers(0, len(CLS_LIST), len(sup)),
                       len(CLS_LIST) + rng.integers(0, max(len(TRI_LIST), 1), len(sup)))
        a = torch.tensor(sup[:, 0], device=GDEV)
        pos = (h[a] * h[NI + torch.tensor(sup[:, 1], device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        ng = (h[a] * h[NI + torch.tensor(neg, device=GDEV)]).sum(-1) * SAGE_LOGIT_SCALE
        loss = TF_.binary_cross_entropy_with_logits(torch.cat([pos, ng]), torch.cat([torch.ones_like(pos), torch.zeros_like(ng)]))
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(loss))
    model.eval()
    with torch.no_grad():
        # suy diễn: khái niệm chỉ nhận tin từ ảnh huấn luyện; mọi ảnh (kể cả P-CHECK) nhận tin từ khái niệm của chính nó
        H = model(xi, xc, mean_adj(E, Etr, NI + NC).to(GDEV)).cpu().numpy()
    return H, dict(loss_first=losses[0], loss_last=losses[-1], n_edges=int(len(E)), n_train_edges=int(len(Etr)))


def query_concepts(parsed):
    E, T = parsed
    return [CON[("c", c)] for c in sorted(E) if ("c", c) in CON] + [CON[("t", t)] for t in sorted(T) if ("t", t) in CON]


def exp1_clip(sp, c):
    ids = SPL1[sp]; caps = [CAPTION[(i, c)] for i in ids]
    T = clip_text(caps); I = np.stack([FEAT1[i] for i in ids])
    topk, cos_top, rel_pos, clip_rank = clip_topk(T @ I.T, K_TOP)
    return dict(ids=ids, topk=topk, zc=zrows(cos_top), rel_pos=rel_pos, clip_rank=clip_rank, parsed=[parse_caption(x) for x in caps])


def sage_components(d, H):
    Q, K = d["topk"].shape
    out = np.zeros((Q, K, 1))
    for q in range(Q):
        cs = query_concepts(d["parsed"][q])
        if not cs: continue
        v = H[NI + np.array(cs)].mean(0); v = v / (np.linalg.norm(v) + 1e-12)
        out[q, :, 0] = H[[IMG_IDX[d["ids"][j]] for j in d["topk"][q]]] @ v
    return out


VARIANTS = {"SAGE": dict(layers=2, rand=False), "SAGE-rand": dict(layers=2, rand=True), "SAGE-0": dict(layers=0, rand=False)}
RES1G = {"seeds": {}}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, cfg in VARIANTS.items():
        H, tr = train_sage(s, **cfg)
        ct, cc = sage_components(tune, H), sage_components(check, H)
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1G["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_concept_coverage=float(np.mean([len(query_concepts(p)) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"G1 SAGE-CLIP": rr["SAGE"] - rr["CLIP"], "G2 SAGE-rand": rr["SAGE"] - rr["SAGE-rand"],
                                                        "G3 SAGE-noedge": rr["SAGE"] - rr["SAGE-0"]}))
    p3_log(f"TN1b seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1G["seeds"][s]["check"].items()},
        "| α", {k: v["alpha"] for k, v in info.items()}, "| loss", {k: (round(v["loss_first"], 3), round(v["loss_last"], 3)) for k, v in info.items()})
RES1G["mean_std"] = {k: {m: mean_std([RES1G["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *VARIANTS]}
RES1G["decision"] = {h: ("DƯƠNG" if all(RES1G["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h in RES1G["seeds"][0]["bootstrap"]}
RES1G.update(n_train_images=len(TRAIN_IMGS), n_eval_images=len(EVAL_IMGS), n_class_nodes=len(CLS_LIST), n_triple_nodes=len(TRI_LIST),
             hyper=dict(dim=SAGE_DIM, epochs=SAGE_EPOCHS, lr=SAGE_LR, wd=SAGE_WD, dropout=SAGE_DROP, msg_frac=SAGE_MSG_FRAC, logit_scale=SAGE_LOGIT_SCALE))
p3_log("TN1b quyết định (đăng ký trước):", RES1G["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1G["mean_std"].items()}, "|", elapsed())


## 6c. Thí nghiệm 1c — SCENIR (ICML 2025), phương pháp từ 2024
Đối chiếu với mã gốc: `final_pipeline/SCENIR_VERIFICATION_VI.md`. Học không giám sát trên scene graph của ảnh huấn luyện; truy vấn = scene graph tách từ caption; ablation cạnh ngẫu nhiên / bỏ cạnh.

In [ ]:
# ============================== 6c. THÍ NGHIỆM 1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (phương pháp từ 2024 cho cấp 2) ==============================
# Xác minh và kế hoạch: final_pipeline/SCENIR_VERIFICATION_VI.md. Cài lại ARGVA của mã gốc (github.com/nickhaidos/scenir-icml2025, MIT) bằng PyTorch thuần:
# encoder GIN 2 nhánh (μ, log σ), feature decoder (MSE), edge decoder MLP + tích vô hướng, discriminator đối kháng; siêu tham số = config.json.
# Huấn luyện KHÔNG giám sát trên scene graph của ảnh huấn luyện (giống GraphSAGE, không chứa P-CHECK). Truy vấn = scene graph tách từ caption.
SC_CFG = dict(epochs=30, lr=1e-3, gamma=0.95, bs=64, latent=1000, layers=3, disc_hidden=32, edge_hidden=256, edge_out=32, disc_steps=5)
SC_EPS, SC_MAX_LOGSTD = 1e-15, 10.0

# ---- đặc trưng đỉnh: Sentence-Transformer 768 chiều như bài báo (giả định all-mpnet-base-v2); dự phòng CLIP-text ViT-L/14 (768) ----
SC_FEAT_SRC = None
if not SMOKE:
    try:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            sh([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"]); from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=GDEV)
        SC_X = torch.tensor(_st.encode(CLS_LIST, convert_to_numpy=True, normalize_embeddings=False), dtype=torch.float32)
        SC_FEAT_SRC = "sentence-transformers/all-mpnet-base-v2"; del _st
    except Exception as ex:
        p3_log("SCENIR: không tải được Sentence-Transformer, dùng CLIP-text:", repr(ex)[:200])
if SC_FEAT_SRC is None:
    SC_X = torch.tensor(clip_text(CLS_LIST), dtype=torch.float32); SC_FEAT_SRC = f"CLIP-text {CLIP_MODEL}" if not SMOKE else "fake (smoke)"
SC_CI = {c: k for k, c in enumerate(CLS_LIST)}
p3_log(f"SCENIR: đặc trưng đỉnh {SC_FEAT_SRC}, {tuple(SC_X.shape)}")


def sg_of_pairs(pairs):
    """đồ thị lớp-level: đỉnh = lớp tham gia ≥ 1 quan hệ (bỏ đỉnh cô lập như bài báo), cạnh vô hướng (lưu hai chiều)."""
    pairs = sorted({tuple(sorted((a, b))) for a, b in pairs if a != b and a in SC_CI and b in SC_CI})
    nodes = sorted({c for p in pairs for c in p})
    loc = {c: k for k, c in enumerate(nodes)}
    ei = [(loc[a], loc[b]) for a, b in pairs] + [(loc[b], loc[a]) for a, b in pairs]
    return [SC_CI[c] for c in nodes], np.array(ei, dtype=np.int64).reshape(-1, 2).T


def image_sg(i, variant, seed):
    pairs = list(GRAPH[i].between.keys())
    nodes, ei = sg_of_pairs(pairs)
    if variant == "noedge":
        return nodes, np.zeros((2, 0), np.int64)
    if variant == "rand" and len(nodes) > 1:
        rr = np.random.default_rng(SEED_RAND + seed * 100003 + int(i))
        allp = [(a, b) for a in range(len(nodes)) for b in range(a + 1, len(nodes))]
        pick = [allp[k] for k in rr.choice(len(allp), size=min(ei.shape[1] // 2, len(allp)), replace=False)]
        ei = np.array([(a, b) for a, b in pick] + [(b, a) for a, b in pick], dtype=np.int64).reshape(-1, 2).T
    return nodes, ei


def query_sg(parsed):
    return sg_of_pairs([(s, o) for (s, p, o) in parsed[1]])


class GIN(nn.Module):
    """GINConv(Linear), ε = 0: h'_v = W·(h_v + Σ_{u→v} h_u)."""
    def __init__(self, a, b):
        super().__init__(); self.lin = nn.Linear(a, b)

    def forward(self, x, ei):
        agg = torch.zeros_like(x).index_add_(0, ei[1], x[ei[0]]) if ei.shape[1] else torch.zeros_like(x)
        return self.lin(x + agg)


class ScenirModel(nn.Module):
    def __init__(self, din):
        super().__init__()
        L, H = SC_CFG["layers"], SC_CFG["latent"]
        dims = [din] + [H] * L
        self.mu = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.ls = nn.ModuleList(GIN(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.disc = nn.Sequential(nn.Linear(H, SC_CFG["disc_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["disc_hidden"], SC_CFG["disc_hidden"]), nn.ReLU(),
                                  nn.Linear(SC_CFG["disc_hidden"], 1))
        self.fdec = nn.Sequential(nn.Linear(H, din), nn.ReLU(), nn.Linear(din, din))
        self.edec = nn.Sequential(nn.Linear(H, SC_CFG["edge_hidden"]), nn.ReLU(), nn.Linear(SC_CFG["edge_hidden"], SC_CFG["edge_out"]))

    @staticmethod
    def _run(layers, x, ei):
        for k, l in enumerate(layers):
            x = l(x, ei)
            if k < len(layers) - 1: x = TF_.relu(x)
        return x

    def encode(self, x, ei):
        self._mu, self._ls = self._run(self.mu, x, ei), self._run(self.ls, x, ei).clamp(max=SC_MAX_LOGSTD)
        return self._mu + torch.randn_like(self._ls) * torch.exp(self._ls) if self.training else self._mu

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.disc(z)) + SC_EPS).mean()

    def disc_loss(self, z):
        real, fake = torch.sigmoid(self.disc(torch.randn_like(z))), torch.sigmoid(self.disc(z.detach()))
        return -torch.log(real + SC_EPS).mean() - torch.log(1 - fake + SC_EPS).mean()

    def kl_loss(self):
        return -0.5 * torch.mean(torch.sum(1 + 2 * self._ls - self._mu ** 2 - self._ls.exp() ** 2, dim=1))

    def recon_loss(self, x, ei, neg):
        z = self.encode(x, ei)
        feat = TF_.mse_loss(self.fdec(z), x)
        zd = self.edec(z)
        dec = lambda e: torch.sigmoid((zd[e[0]] * zd[e[1]]).sum(-1))
        pos = -torch.log(dec(ei) + SC_EPS).mean() if ei.shape[1] else torch.zeros((), device=x.device)
        ngl = -torch.log(1 - dec(neg) + SC_EPS).mean() if neg.shape[1] else torch.zeros((), device=x.device)
        return pos + ngl + feat


def sc_batch(graphs):
    xs, eis, ptr, off = [], [], [0], 0
    for nodes, ei in graphs:
        xs.append(SC_X[nodes]); eis.append(ei + off); off += len(nodes); ptr.append(off)
    return torch.cat(xs), torch.tensor(np.concatenate(eis, axis=1) if eis else np.zeros((2, 0), np.int64)), np.array(ptr)


def sc_negatives(ei, ptr, rng):
    """cặp (u, v) cùng đồ thị, không tự nối, không trùng cạnh thật; số lượng = số cạnh dương (như batched_negative_sampling)."""
    n = ei.shape[1]
    if n == 0: return np.zeros((2, 0), np.int64)
    exist = set(map(tuple, ei.T.tolist()))
    g = np.searchsorted(ptr, ei[0], side="right") - 1
    out = []
    for gi in rng.choice(g, size=n):
        lo, hi = ptr[gi], ptr[gi + 1]
        for _ in range(10):
            u, v = rng.integers(lo, hi, 2)
            if u != v and (u, v) not in exist: out.append((u, v)); break
    return np.array(out, dtype=np.int64).reshape(-1, 2).T


def train_scenir(seed, variant):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    G = [image_sg(i, variant, seed) for i in TRAIN_IMGS]
    G = [g for g in G if len(g[0]) > 0]
    model = ScenirModel(SC_X.shape[1]).to(GDEV)
    d_opt = torch.optim.AdamW(model.disc.parameters(), lr=SC_CFG["lr"])
    e_opt = torch.optim.AdamW(list(model.mu.parameters()) + list(model.ls.parameters()) + list(model.fdec.parameters()) + list(model.edec.parameters()), lr=SC_CFG["lr"])
    sched = torch.optim.lr_scheduler.ExponentialLR(e_opt, gamma=SC_CFG["gamma"])
    hist = []
    for ep in range(SC_CFG["epochs"]):
        model.train(); perm = rng.permutation(len(G)); tot = 0.0
        for s in range(0, len(G), SC_CFG["bs"]):
            x, ei, ptr = sc_batch([G[k] for k in perm[s:s + SC_CFG["bs"]]])
            neg = torch.tensor(sc_negatives(ei.numpy(), ptr, rng)).to(GDEV); x, ei = x.to(GDEV), ei.to(GDEV)
            e_opt.zero_grad()
            z = model.encode(x, ei)
            for _ in range(SC_CFG["disc_steps"]):
                d_opt.zero_grad(); dl = model.disc_loss(z); dl.backward(); d_opt.step()
            loss = (1 / 6) * model.reg_loss(z) + 3 * model.recon_loss(x, ei, neg) + (1 / 3) * (1 / x.shape[0]) * model.kl_loss()
            loss.backward(); e_opt.step(); tot += float(loss)
        sched.step(); hist.append(tot)
    model.eval()
    return model, dict(n_train_graphs=len(G), loss_first=hist[0], loss_last=hist[-1])


@torch.no_grad()
def sc_embed(model, graphs):
    """nhúng đồ thị = Σ μ của các đỉnh (sum-pool), chuẩn hoá L2; đồ thị rỗng → vector 0."""
    out = np.zeros((len(graphs), SC_CFG["latent"]), np.float32)
    for k, (nodes, ei) in enumerate(graphs):
        if len(nodes) == 0: continue
        z = model.encode(SC_X[nodes].to(GDEV), torch.tensor(ei).to(GDEV)).sum(0)
        out[k] = (z / (z.norm() + 1e-12)).cpu().numpy()
    return out


SC_VARIANTS = {"SCENIR": "full", "SCENIR-rand": "rand", "SCENIR-noedge": "noedge"}
RES1S = {"seeds": {}, "node_features": SC_FEAT_SRC, "config": SC_CFG}
for s in SEEDS:
    tune, check = exp1_clip("P-TUNE", s), exp1_clip("P-CHECK", s)
    ranks, info = {"CLIP": check["clip_rank"]}, {}
    for name, var in SC_VARIANTS.items():
        model, tr = train_scenir(s, var)
        emb_img = dict(zip(SPL1["P-TUNE"] + SPL1["P-CHECK"],
                           sc_embed(model, [image_sg(i, var, s) for i in SPL1["P-TUNE"] + SPL1["P-CHECK"]])))
        qsg = lambda d: sc_embed(model, [query_sg(p) if var != "noedge" else (query_sg(p)[0], np.zeros((2, 0), np.int64)) for p in d["parsed"]])

        def comps_of(d, qe):
            Q, K = d["topk"].shape
            return np.stack([np.stack([emb_img[d["ids"][j]] for j in d["topk"][q]]) @ qe[q] for q in range(Q)])[:, :, None]
        ct, cc = comps_of(tune, qsg(tune)), comps_of(check, qsg(check))
        grid = tune_grid(tune["zc"], ct, tune["rel_pos"], tune["clip_rank"], [(1.0,)], ALPHAS)
        (w, a) = select_params(grid)
        ranks[name] = rank_of_relevant(fuse_scores(check["zc"], cc, w, a), check["rel_pos"], check["clip_rank"])
        info[name] = dict(alpha=a, tune_MRR=grid[(w, a)], alpha_curve_tune_MRR={aa: grid[((1.0,), aa)] for aa in ALPHAS}, **tr)
        del model
    rr = {k: 1.0 / v for k, v in ranks.items()}
    RES1S["seeds"][s] = dict(train=info, check={k: metrics_from_ranks(v) for k, v in ranks.items()},
                             query_graph_coverage=float(np.mean([len(query_sg(p)[0]) > 0 for p in check["parsed"]])),
                             bootstrap=bootstrap_delta({"S1 SCENIR-CLIP": rr["SCENIR"] - rr["CLIP"], "S2 SCENIR-rand": rr["SCENIR"] - rr["SCENIR-rand"],
                                                        "S3 SCENIR-noedge": rr["SCENIR"] - rr["SCENIR-noedge"]}))
    p3_log(f"TN1c seed {s}: MRR", {k: round(v["MRR"], 4) for k, v in RES1S["seeds"][s]["check"].items()}, "| α", {k: v["alpha"] for k, v in info.items()},
        "| loss", {k: (round(v["loss_first"], 2), round(v["loss_last"], 2)) for k, v in info.items()})
RES1S["mean_std"] = {k: {m: mean_std([RES1S["seeds"][s]["check"][k][m] for s in SEEDS]) for m in ("R1", "R5", "R10", "MRR", "nDCG10")}
                     for k in ["CLIP", *SC_VARIANTS]}
RES1S["decision"] = {h: ("DƯƠNG" if all(RES1S["seeds"][s]["bootstrap"][h]["lo"] > 0 for s in SEEDS) else "KHÔNG DƯƠNG") for h in RES1S["seeds"][0]["bootstrap"]}
p3_log("TN1c quyết định (đăng ký trước):", RES1S["decision"], "| MRR:", {k: round(v["MRR"]["mean"], 4) for k, v in RES1S["mean_std"].items()}, "|", elapsed())


## 7. Thí nghiệm 2 — benchmark kết hợp công khai (ARO, SugarCrepe)

In [ ]:
# ============================== 7. THÍ NGHIỆM 2 — Benchmark kết hợp công khai (ARO VG-Relation, SugarCrepe): ảnh → chọn caption đúng ==============================
# Đăng ký trước: v6_compositional_benchmarks/PREREGISTRATION_VI.md. Đồ thị = scene graph RelTR suy từ điểm ảnh (KHÔNG dùng GT vì caption ARO sinh từ GT).
# Đối chứng: C-rand (đặc trưng mục khác), C-shuf (hoán vị vị từ), C-nogeo (bỏ vị trí). α chọn trên TUNE, đánh giá CHECK, 3 seed chia.
FEATS2 = reltr_clip_features(ITEMS2)
TT2, TF2 = clip_text([it["true"] for it in ITEMS2]), clip_text([it["false"] for it in ITEMS2])
V3B = set(V3_BENCH_IDS)
p3_log(f"TN2: đặc trưng {len(FEATS2)} ảnh | {elapsed()}")

RAND2 = {}
for b in ("aro", "replace_rel", "swap_obj"):
    idx = [n for n, it in enumerate(ITEMS2) if it["bench"] == b]
    perm = rand_permutation(len(idx))
    for j, n in enumerate(idx): RAND2[n] = idx[perm[j]]

ROWS2 = []
for n, it in enumerate(ITEMS2):
    F, Fr = FEATS2[n], FEATS2[RAND2[n]]
    row = dict(bench=it["bench"], item=it["item"], key=it["key"], subset=it["subset"], rel=it["rel"],
               dc=float(F["img"] @ TT2[n] - F["img"] @ TF2[n]), in_v3_bench=bool(it["vg_id"] in V3B))
    if it["bench"] == "aro":
        pt, pf = aro_parse(it["true"]), aro_parse(it["false"])
        valid = pt[1] != "other" and pf[1] != "other"
        if valid:
            prs = shuffled_pr(F, it["key"])
            gt, et = aro_graph_score(F, pt); gf, ef = aro_graph_score(F, pf)
            row.update(dg=gt - gf, dg_rand=aro_graph_score(Fr, pt)[0] - aro_graph_score(Fr, pf)[0],
                       dg_shuf=(aro_graph_score(F, pt, pr=prs)[0] - aro_graph_score(F, pf, pr=prs)[0]) if pt[1] == "pred" else np.nan,
                       dg_nogeo=(aro_graph_score(F, pt, nogeo=True)[0] - aro_graph_score(F, pf, nogeo=True)[0]) if pt[1] == "geo" else np.nan,
                       ev_true=json.dumps(et), ev_false=json.dumps(ef))
    else:
        Tt, Tf = parse_caption(it["true"])[1], parse_caption(it["false"])[1]
        valid = bool(Tt) and bool(Tf) and Tt != Tf
        if valid:
            prs = shuffled_pr(F, int(it["key"][4:]))
            gt, et = triples_graph_score(F, Tt); gf, ef = triples_graph_score(F, Tf)
            if gt is None or gf is None: valid = False
            else:
                row.update(dg=gt - gf, dg_rand=triples_graph_score(Fr, Tt)[0] - triples_graph_score(Fr, Tf)[0],
                           dg_shuf=triples_graph_score(F, Tt, pr=prs)[0] - triples_graph_score(F, Tf, pr=prs)[0], dg_nogeo=np.nan,
                           ev_true=json.dumps(et), ev_false=json.dumps(ef))
    row["valid"] = bool(valid)
    for k in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(k, np.nan)
    ROWS2.append(row)


def exp2_evaluate(rows, s):
    sp = np.array([v6_split(r["key"], s) for r in rows]); ck, tu = sp == "CHECK", sp == "TUNE"
    dc = np.array([r["dc"] for r in rows]); valid = np.array([r["valid"] for r in rows]); clusters = np.array([r["key"] for r in rows])
    out = dict(n_tune=int(tu.sum()), n_check=int(ck.sum()), coverage=float(valid.mean()))
    res = {"CLIP": correct(dc)}
    for var in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"):
        dg = np.array([r[var] for r in rows], dtype=float); v = valid & ~np.isnan(dg)
        if not v.any(): continue
        dgz = np.where(v, dg, 0.0)
        res[f"graph_only[{var}]"] = np.where(v, correct(dgz), 0.5)
        a, sc, sg, acc = tune_alpha(dc[tu], dgz[tu], v[tu])
        res[f"fusion[{var}]"] = correct(fuse2(dc, dgz, v, a, sc, sg)); out[f"alpha[{var}]"] = a
        if var == "dg": out["alpha_grid_tune"] = acc
    out["acc_check"] = {k: float(v[ck].mean()) for k, v in res.items()}
    out["acc_check_valid_only"] = {k: float(v[ck & valid].mean()) for k, v in res.items()} if (ck & valid).any() else {}
    out["n_check_valid"] = int((ck & valid).sum())
    comps = {"H fusion-CLIP": ("fusion[dg]", "CLIP"), "graph-rand": ("graph_only[dg]", "graph_only[dg_rand]"),
             "fusion-fusion_rand": ("fusion[dg]", "fusion[dg_rand]"), "graph-shuf": ("graph_only[dg]", "graph_only[dg_shuf]"),
             "graph-nogeo": ("graph_only[dg]", "graph_only[dg_nogeo]"), "graph-CLIP": ("graph_only[dg]", "CLIP")}
    out["bootstrap_check"] = {}
    for nm, (a, b) in comps.items():
        if a in res and b in res:
            m = ck.copy()
            if "shuf" in nm: m &= ~np.isnan(np.array([r["dg_shuf"] for r in rows], dtype=float)) & valid
            if "nogeo" in nm: m &= ~np.isnan(np.array([r["dg_nogeo"] for r in rows], dtype=float)) & valid
            if m.sum() > 1: out["bootstrap_check"][nm] = dict(n=int(m.sum()), **bootstrap_diff_by_cluster(res[a][m], res[b][m], clusters[m]))
    return out


GROUPS2 = {"ARO_all": lambda r: r["bench"] == "aro", "ARO_geo": lambda r: r["bench"] == "aro" and r["subset"] == "geo",
           "ARO_pred": lambda r: r["bench"] == "aro" and r["subset"] == "pred", "ARO_other": lambda r: r["bench"] == "aro" and r["subset"] == "other",
           "ARO_all_excl_v3bench": lambda r: r["bench"] == "aro" and not r["in_v3_bench"],
           "SC_replace_rel": lambda r: r["bench"] == "replace_rel", "SC_swap_obj": lambda r: r["bench"] == "swap_obj"}
RES2 = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2 if f(r)], s) for g, f in GROUPS2.items() if any(f(r) for r in ROWS2)} for s in SEEDS}}
RES2["mean_std"] = {g: {m: mean_std([RES2["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2["seeds"][0][g]["acc_check"]}
                    for g in RES2["seeds"][0]}
RES2["decision"] = {g: ("DƯƠNG" if all(RES2["seeds"][s][g]["bootstrap_check"].get("H fusion-CLIP", {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                    for g in ("ARO_all", "SC_replace_rel") if g in RES2["seeds"][0]}
RES2.update(items=dict(Counter(i["bench"] for i in ITEMS2)), excluded=dict(EXCL2), overlap_v3_benchmark_items=sum(it["vg_id"] in V3B for it in ITEMS2))
for g in RES2["mean_std"]:
    p3_log(f"TN2 {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "fusion[dg_rand]")})
p3_log("TN2 quyết định (đăng ký trước):", RES2["decision"], "|", elapsed())


## 7b. Thí nghiệm 2b — Cấp 4 (a): cạnh hình học trên COCO-spatial
Đăng ký trước: `final_pipeline/LEVEL4A_PREREGISTRATION_VI.md`. Cạnh trái / phải / trên / dưới suy từ box RelTR; đối chứng bỏ vị trí, đặc trưng mục khác, lật ảnh.

In [ ]:
# ============================== 7b. THÍ NGHIỆM 2b — Cấp 4 hướng (a): cạnh hình học trên tập công khai thứ hai (COCO-spatial, What'sUp EMNLP 2023) ==============================
# Đăng ký trước: final_pipeline/LEVEL4A_PREREGISTRATION_VI.md. Cùng RelTR, CLIP, cách kết hợp và chọn α như TN2. Lọc rò rỉ như SugarCrepe.
CSP_GDRIVE = "1TCEoM0mgFmz8T4cF7PQ3XJmO6JjtiQ-s"
CSP_PATH = BENCH["cocosp"] if SMOKE else fetch(f"{WORK}/coco_qa_two_obj.json", gdrive=CSP_GDRIVE)
CSP_SHA = sha256(CSP_PATH)
CSP_RX = re.compile(r"^A photo of an? (.+?) (?:to the )?(left of|right of|above|below) (?:an? |the )?(.+)$")
CSP_DIR = {"left of": ("x", 1), "right of": ("x", -1), "above": ("y", 1), "below": ("y", -1)}


def csp_name(phrase):
    return norm_name(phrase) or norm_name(phrase.split()[-1])


def csp_parse(cap):
    m = CSP_RX.match(cap)
    if not m: return None
    X, Y = csp_name(m.group(1)), csp_name(m.group(3))
    return (X, m.group(2), Y) if X and Y else ("?", m.group(2), "?")


def geo_axis_score(F, X, Y, rel, nogeo=False):
    """như geo_score nhưng cho cả trục ngang (left/right) và trục dọc (above/below): σ(sign·(c_v − c_u)/τ)."""
    axis, sign = CSP_DIR[rel]
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    a, b, c = P[:, CLS_IDX[X]], P[:, CLS_IDX[Y]], B[:, 0 if axis == "x" else 1]
    s = np.ones((len(c), len(c))) if nogeo else 1.0 / (1.0 + np.exp(-sign * (c[None, :] - c[:, None]) / TAU))
    M = a[:, None] * b[None, :] * s; np.fill_diagonal(M, 0.0)
    u, v = np.unravel_index(int(np.argmax(M)), M.shape)
    return float(np.log(max(M[u, v], FLOOR))), dict(kind="geo", u=int(u), v=int(v), p=float(M[u, v]), x_box=[float(x) for x in B[u]], y_box=[float(x) for x in B[v]])


ITEMS2B, EXCL2B = [], Counter()
for k, x in enumerate(json.load(open(CSP_PATH, encoding="utf-8"))):
    coco = int(x[0]); vid = VG_OF_COCO.get(coco)
    if vid is not None and vid not in VALSET: EXCL2B["vg_not_VAL"] += 1; continue
    path = os.path.join(P["coco_val"], f"{coco:012d}.jpg")
    if not os.path.exists(path): EXCL2B["no_jpg"] += 1; continue
    pt = csp_parse(x[1]); pf = csp_parse(x[2])
    if pt is None or pf is None: EXCL2B["unparsed"] += 1; continue
    ITEMS2B.append(dict(bench="cocosp", item=f"cocosp{k}", key=f"cocosp{coco}", vg_id=vid, path=path, crop=None, true=x[1], false=x[2],
                        pt=pt, pf=pf, axis=CSP_DIR[pt[1]][0], valid=pt[0] != "?" and pf[0] != "?"))
GUARD_VAL.check([it["vg_id"] for it in ITEMS2B if it["vg_id"] is not None], "COCO-spatial")
p3_log(f"TN2b COCO-spatial: {len(ITEMS2B)} mục giữ lại, loại {dict(EXCL2B)}, đồ thị chấm được {sum(it['valid'] for it in ITEMS2B)} | {elapsed()}")

FEATS2B = reltr_clip_features(ITEMS2B)
T2B, F2B = clip_text([it["true"] for it in ITEMS2B]), clip_text([it["false"] for it in ITEMS2B])
PERM2B = rand_permutation(len(ITEMS2B))
_flip_idx = [n for n, it in enumerate(ITEMS2B) if it["valid"]]
FLIP2B = {}
if _flip_idx:
    _hidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "x"]; _vidx = [n for n in _flip_idx if ITEMS2B[n]["axis"] == "y"]
    FLIP2B.update(dict(zip(_hidx, reltr_clip_features([ITEMS2B[n] for n in _hidx], flip=True))))
    if _vidx:
        _orig_pil_of = pil_of
        if not SMOKE:
            pil_of = lambda it, flip=False: ImageOps.flip(pil_path(it["path"])) if flip else pil_path(it["path"])     # lật DỌC cho mục trên/dưới
        FLIP2B.update(dict(zip(_vidx, reltr_clip_features([ITEMS2B[n] for n in _vidx], flip=True))))
        pil_of = _orig_pil_of
p3_log(f"TN2b: đặc trưng {len(FEATS2B)} ảnh + {len(FLIP2B)} ảnh lật | {elapsed()}")

ROWS2B = []
for n, it in enumerate(ITEMS2B):
    F, Fr = FEATS2B[n], FEATS2B[PERM2B[n]]
    row = dict(item=it["item"], key=it["key"], axis=it["axis"], rel=it["pt"][1], valid=it["valid"], dc=float(F["img"] @ T2B[n] - F["img"] @ F2B[n]))
    if it["valid"]:
        (X, r, Y), (Xf, rf, Yf) = it["pt"], it["pf"]
        gt, et = geo_axis_score(F, X, Y, r); gf, ef = geo_axis_score(F, Xf, Yf, rf)
        row.update(dg=gt - gf, dg_rand=geo_axis_score(Fr, X, Y, r)[0] - geo_axis_score(Fr, Xf, Yf, rf)[0],
                   dg_nogeo=geo_axis_score(F, X, Y, r, nogeo=True)[0] - geo_axis_score(F, Xf, Yf, rf, nogeo=True)[0],
                   dg_shuf=np.nan, ev_true=json.dumps(et), ev_false=json.dumps(ef))
        if n in FLIP2B:
            Ff = FLIP2B[n]
            dgf = geo_axis_score(Ff, X, Y, r)[0] - geo_axis_score(Ff, Xf, Yf, rf)[0]
            dcf = float(Ff["img"] @ T2B[n] - Ff["img"] @ F2B[n])
            row.update(flip_graph_reversed=float(np.sign(dgf) == -np.sign(row["dg"]) and row["dg"] != 0),
                       flip_clip_reversed=float(np.sign(dcf) == -np.sign(row["dc"]) and row["dc"] != 0))
    for kk in ("dg", "dg_rand", "dg_shuf", "dg_nogeo"): row.setdefault(kk, np.nan)
    ROWS2B.append(row)

GROUPS2B = {"COCOSP_all": lambda r: True, "COCOSP_leftright": lambda r: r["axis"] == "x", "COCOSP_abovebelow": lambda r: r["axis"] == "y"}
RES2B = {"seeds": {s: {g: exp2_evaluate([r for r in ROWS2B if f(r)], s) for g, f in GROUPS2B.items() if any(f(r) for r in ROWS2B)} for s in SEEDS}}
RES2B["mean_std"] = {g: {m: mean_std([RES2B["seeds"][s][g]["acc_check"][m] for s in SEEDS]) for m in RES2B["seeds"][0][g]["acc_check"]}
                     for g in RES2B["seeds"][0]}
RES2B["decision"] = {h: ("DƯƠNG" if all(RES2B["seeds"][s]["COCOSP_all"]["bootstrap_check"].get(k, {}).get("lo", -1) > 0 for s in SEEDS) else "KHÔNG DƯƠNG")
                     for h, k in (("A1 fusion-CLIP", "H fusion-CLIP"), ("A2 graph-nogeo", "graph-nogeo"), ("A3 graph-rand", "graph-rand"))}
_fl = [r for r in ROWS2B if "flip_graph_reversed" in r]
if _fl:
    fg, fc = np.array([r["flip_graph_reversed"] for r in _fl]), np.array([r["flip_clip_reversed"] for r in _fl])
    RES2B["counterfactual"] = dict(n=len(_fl), graph_reversed=float(fg.mean()), clip_reversed=float(fc.mean()),
                                   gap=bootstrap_diff_by_cluster(fg, fc, [r["key"] for r in _fl]))
RES2B.update(n_items=len(ITEMS2B), excluded=dict(EXCL2B), n_scorable=int(sum(it["valid"] for it in ITEMS2B)), sha256=CSP_SHA)
for g in RES2B["mean_std"]:
    p3_log(f"TN2b {g}:", {k: round(v["mean"] * 100, 1) for k, v in RES2B["mean_std"][g].items() if k in ("CLIP", "graph_only[dg]", "fusion[dg]", "graph_only[dg_rand]", "graph_only[dg_nogeo]")})
p3_log("TN2b quyết định (đăng ký trước):", RES2B["decision"], "| phản thực tế:", RES2B.get("counterfactual"), "|", elapsed())


## 8. Giải thích (XAI, cấp 3)
Đăng ký trước: `v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md`. Sparsity gồm kích thước bằng chứng (hằng số theo thiết kế) và **độ tập trung bằng chứng** (thay đổi theo mục).

In [ ]:
# ============================== 8. GIẢI THÍCH (XAI, cấp 3): fidelity · sufficiency · phản thực tế (lật ảnh) · validity (khớp GT) · sparsity ==============================
# Đăng ký trước: v6_compositional_benchmarks/XAI_PREREGISTRATION_VI.md. Đơn vị: mục ARO đồ thị chấm được, phần CHECK của seed 0.
# Giải thích = đúng phần tử tạo ra điểm đồ thị: truy vấn bộ ba j* (vị từ) hoặc cặp hộp u*, v* (trái/phải).
XI = [n for n, it in enumerate(ITEMS2) if it["bench"] == "aro" and aro_parse(it["true"])[1] != "other" and aro_parse(it["false"])[1] != "other"
      and v6_split(it["key"], 0) == "CHECK"]
XI.sort(key=lambda n: int(ITEMS2[n]["item"][3:]))
GEO_N = [n for n in XI if aro_parse(ITEMS2[n]["true"])[1] == "geo"]
FLIP = dict(zip(GEO_N, reltr_clip_features([ITEMS2[n] for n in GEO_N], flip=True)))
p3_log(f"XAI: {len(XI)} mục ({len(GEO_N)} trái/phải) | đặc trưng ảnh lật xong | {elapsed()}")


def zero_rows(F, kind, rows):
    G = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in F.items()}
    for r in rows:
        if kind == "pred": G["ps"][r] = 0; G["po"][r] = 0; G["pr"][r] = 0
        else: (G["ps"] if r < NQ else G["po"])[r % NQ] = 0
    return G


def keep_rows(F, kind, rows):
    return zero_rows(F, kind, [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows)])


ev_rows = lambda e: [e["j"]] if e["kind"] == "pred" else [e["u"], e["v"]]
sgn = lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
cxcywh = lambda b: (b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2)


def dgraph(F, it):
    gt, et = aro_graph_score(F, aro_parse(it["true"])); gf, ef = aro_graph_score(F, aro_parse(it["false"]))
    return gt - gf, et, ef


def evidence_concentration(F, parsed):
    """sparsity không cố định: tỷ lệ điểm của phần tử giải thích (max) trên tổng điểm mọi ứng viên (truy vấn bộ ba hoặc cặp hộp).
    Gần 1 ⇒ giải thích gọn (một bằng chứng áp đảo); gần 0 ⇒ bằng chứng dàn trải trên nhiều ứng viên."""
    X, kind, r, Y = parsed
    if kind == "pred":
        v = F["ps"][:, CLS_IDX[X]].astype(np.float64) * F["pr"][:, PRED_IDX[r]].astype(np.float64) * F["po"][:, CLS_IDX[Y]].astype(np.float64)
        return float(v.max() / max(v.sum(), FLOOR))
    P = np.concatenate([F["ps"], F["po"]]).astype(np.float64); B = np.concatenate([F["sb"], F["ob"]]).astype(np.float64)
    diff = (B[None, :, 0] - B[:, None, 0]) if r == "L" else (B[:, None, 0] - B[None, :, 0])
    M = P[:, CLS_IDX[X]][:, None] * P[:, CLS_IDX[Y]][None, :] / (1.0 + np.exp(-diff / TAU)); np.fill_diagonal(M, 0.0)
    return float(M.max() / max(M.sum(), FLOOR))


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix * iy
    return ix * iy / u if u > 0 else 0.0


def crop_norm(gt, crop):
    cw, ch = crop[2] - crop[0], crop[3] - crop[1]
    return ((gt[0] - crop[0]) / cw, (gt[1] - crop[1]) / ch, (gt[2] - crop[0]) / cw, (gt[3] - crop[1]) / ch)


ROWS3 = []
for n in XI:
    it, F = ITEMS2[n], FEATS2[n]
    kind = aro_parse(it["true"])[1]
    dg, et, ef = dgraph(F, it)
    dc = float(F["img"] @ TT2[n] - F["img"] @ TF2[n])
    row = dict(item=it["item"], n=n, vg_id=it["vg_id"], kind=kind, true=it["true"], false=it["false"], dg=dg, dc=dc,
               graph_correct=float(correct([dg])[0]), clip_correct=float(correct([dc])[0]), ev_true=json.dumps(et), ev_false=json.dumps(ef))
    if dg != 0:
        rows_ev = ev_rows(et if dg > 0 else ef)
        row["del_changed"] = float(sgn(dgraph(zero_rows(F, kind, rows_ev), it)[0]) != sgn(dg))
        pool = [r for r in range(NQ if kind == "pred" else 2 * NQ) if r not in set(rows_ev)]
        rr = np.random.default_rng(int(hashlib.md5(it["item"].encode()).hexdigest(), 16) % 2 ** 32)
        row["rand_changed"] = float(np.mean([sgn(dgraph(zero_rows(F, kind, list(rr.choice(pool, size=len(rows_ev), replace=False))), it)[0]) != sgn(dg)
                                             for _ in range(N_RAND_DEL)]))
        row["suff_kept"] = float(sgn(dgraph(keep_rows(F, kind, sorted(set(ev_rows(et)) | set(ev_rows(ef)))), it)[0]) == sgn(dg))
        row["evidence_fraction"] = len(rows_ev) / (NQ if kind == "pred" else 2 * NQ)
        row["evidence_concentration"] = evidence_concentration(F, aro_parse(it["true"] if dg > 0 else it["false"]))
        if dg > 0 and it["xid"] in GTBOX and it["yid"] in GTBOX:
            gx, gy = crop_norm(GTBOX[it["xid"]], it["crop"]), crop_norm(GTBOX[it["yid"]], it["crop"])
            bx_, by_ = (et["sub_box"], et["obj_box"]) if kind == "pred" else (et["x_box"], et["y_box"])
            row["iou_x"], row["iou_y"] = iou(cxcywh(bx_), gx), iou(cxcywh(by_), gy)
            row["valid_both"] = float(row["iou_x"] >= 0.5 and row["iou_y"] >= 0.5); row["valid_any"] = float(row["iou_x"] >= 0.5 or row["iou_y"] >= 0.5)
    if n in FLIP:
        dgf = dgraph(FLIP[n], it)[0]; dcf = float(FLIP[n]["img"] @ TT2[n] - FLIP[n]["img"] @ TF2[n])
        row["flip_graph_reversed"] = float(sgn(dgf) == -sgn(dg) and dg != 0); row["flip_clip_reversed"] = float(sgn(dcf) == -sgn(dc) and dc != 0)
    ROWS3.append(row)


def _col(name, rows):
    return np.array([r[name] for r in rows if name in r], float), [r["vg_id"] for r in rows if name in r]


RES3 = {}
for kind in ("all", "geo", "pred"):
    sub = [r for r in ROWS3 if kind == "all" or r["kind"] == kind]
    d, cl = _col("del_changed", sub); rnd, _ = _col("rand_changed", sub)
    o = dict(n=len(sub), graph_acc=float(np.mean([r["graph_correct"] for r in sub])), clip_acc=float(np.mean([r["clip_correct"] for r in sub])))
    if len(d):
        vb, _ = _col("valid_both", sub); va, _ = _col("valid_any", sub)
        o.update(fidelity_del_changed=float(d.mean()), fidelity_rand_changed=float(rnd.mean()), fidelity_gap=bootstrap_diff_by_cluster(d, rnd, cl),
                 sufficiency_kept=float(_col("suff_kept", sub)[0].mean()), sparsity_evidence_fraction=float(_col("evidence_fraction", sub)[0].mean()),
                 sparsity_concentration_mean=float(_col("evidence_concentration", sub)[0].mean()),
                 sparsity_concentration_median=float(np.median(_col("evidence_concentration", sub)[0])),
                 validity_both_iou50=float(vb.mean()) if len(vb) else None, validity_any_iou50=float(va.mean()) if len(va) else None, n_validity=len(vb))
    fg, fcl = _col("flip_graph_reversed", sub); fc, _ = _col("flip_clip_reversed", sub)
    if len(fg):
        o.update(counterfactual_graph_reversed=float(fg.mean()), counterfactual_clip_reversed=float(fc.mean()), n_counterfactual=len(fg),
                 counterfactual_gap=bootstrap_diff_by_cluster(fg, fc, fcl))
    RES3[kind] = o
p3_log("XAI:", json.dumps({k: {m: (round(v, 3) if isinstance(v, float) else v) for m, v in o.items() if not isinstance(v, dict)} for k, o in RES3.items()}, ensure_ascii=False)[:900])

# ---- 10 ví dụ (5 đồ thị đúng, 5 đồ thị sai; 5 mục đầu theo id — luật khai báo trước) ----
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
EX3 = [r for r in ROWS3 if r["graph_correct"] == 1.0][:5] + [r for r in ROWS3 if r["graph_correct"] == 0.0][:5]


def draw_explanation(r, path):
    it = ITEMS2[r["n"]]; im = pil_of(it); W, H = im.size
    fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(im); ax.axis("off")
    for ev, color, lab in ((json.loads(r["ev_true"]), "lime", "đúng"), (json.loads(r["ev_false"]), "red", "sai")):
        bxs = [ev["sub_box"], ev["obj_box"]] if ev["kind"] == "pred" else [ev["x_box"], ev["y_box"]]
        for b, nm in zip(bxs, ("X", "Y")):
            x0, y0, x1, y1 = cxcywh(b)
            ax.add_patch(mpatches.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H, fill=False, ec=color, lw=2, ls="-" if lab == "đúng" else "--"))
            ax.text(x0 * W, y0 * H, f"{nm}({lab})", color=color, fontsize=8, va="bottom")
    ax.set_title(f"{r['item']} | {'ĐỒ THỊ ĐÚNG' if r['graph_correct'] == 1 else 'ĐỒ THỊ SAI'}\nđúng: {r['true']} (p={json.loads(r['ev_true'])['p']:.2e})\n"
                 f"sai: {r['false']} (p={json.loads(r['ev_false'])['p']:.2e})\nΔCLIP={r['dc']:+.3f}", fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


for k, r in enumerate(EX3): draw_explanation(r, f"{FIG}/xai_ex{k:02d}_{r['item']}.png")
p3_log(f"XAI: đã vẽ {len(EX3)} ví dụ vào {FIG} | {elapsed()}")


## 10. Demo: nhập truy vấn → top-k + giải thích
Gọi `show(search("a child holding a kite"))` để thử truy vấn khác.

In [ ]:
# ============================== 10. DEMO: nhập truy vấn → top-k ảnh + quan hệ/đường đi giải thích (đề cương mục 8) ==============================
# Dùng gallery P-CHECK của TN1 (ảnh VG có scene graph GT), tham số (w, α) đã chọn trên P-TUNE (seed 0). Không tinh chỉnh gì thêm.
DEMO_IDS = EXP1_KEEP["check"]["ids"]
DEMO_I = EXP1_KEEP["check"]["I"]
(DEMO_W, DEMO_A), _ = EXP1_KEEP["params"]["G-full"]


def search(query, k=5, alpha=None, w=None):
    """CLIP top-K theo cosine → điểm đồ thị (3 kiểu đường đi) → xếp hạng lại α·z(cos) + (1−α)·z(graph). Trả về top-k + giải thích."""
    alpha = DEMO_A if alpha is None else alpha; w = DEMO_W if w is None else w
    t = clip_text([query])[0]
    cos = DEMO_I @ t
    top = np.argsort(-cos, kind="stable")[:K_TOP]
    E, T = parse_caption(query)
    comps = np.array([[(lambda d: (d["s_rel"], d["s_pair"], d["s_obj"]))(match_detail(T, E, GRAPH[DEMO_IDS[j]])) for j in top]])
    final = fuse_scores(zrows(cos[top][None, :]), comps, w, alpha)[0]
    order = np.lexsort((np.arange(len(top)), -final))[:k]
    res = []
    for r, kk in enumerate(order):
        i = DEMO_IDS[top[kk]]; d = match_detail(T, E, GRAPH[i])
        res.append(dict(rank=r + 1, clip_rank=int(kk) + 1, image_id=i, score=float(final[kk]), cos=float(cos[top[kk]]),
                        matched_edges=d["edges"], matched_pairs=d["pairs"], matched_nodes=d["nodes"]))
    return dict(query=query, parsed_entities=sorted(E), parsed_triples=sorted("|".join(x) for x in T), alpha=alpha, w=list(w), results=res)


def show(out, path=None):
    n = len(out["results"])
    fig, axs = plt.subplots(1, n, figsize=(3.3 * n, 3.9)); axs = np.atleast_1d(axs)
    for ax, r in zip(axs, out["results"]):
        ax.imshow(pil_path(VG_JPG[r["image_id"]])); ax.axis("off")
        why = "; ".join(e.split("<=")[0] for e in r["matched_edges"]) or ("cặp: " + ", ".join(r["matched_pairs"]) if r["matched_pairs"] else "đối tượng: " + ", ".join(r["matched_nodes"]))
        ax.set_title(f"#{r['rank']} (CLIP #{r['clip_rank']}) id {r['image_id']}\n{why[:60]}", fontsize=7)
    fig.suptitle(f"'{out['query']}' | bộ ba: {out['parsed_triples']} | α={out['alpha']}", fontsize=8); fig.tight_layout()
    path = path or f"{FIG}/_last_search.png"
    fig.savefig(path, dpi=100); plt.close(fig)
    try:
        from IPython.display import Image as _IImg, display
        display(_IImg(filename=path))                       # hiện hình khi chạy trong notebook
    except Exception:
        pass


DEMO = []
for q in ["a man riding a horse on the beach", "a dog sitting on a bench", "a woman holding an umbrella on the street"]:
    out = search(q); DEMO.append(out); show(out, f"{FIG}/demo_{len(DEMO)}.png")
    p3_log(f"DEMO '{q}':", [(r["image_id"], r["clip_rank"], r["matched_edges"][:1]) for r in out["results"]])
# Dùng tương tác trong notebook:  show(search("a child holding a kite"))


## 10b. Năm truy vấn thành công và năm truy vấn thất bại (văn bản → ảnh)
Luật chọn cố định; phân tích chi tiết trong `final_pipeline/CASE_ANALYSIS_LEVEL1_VI.md`.

In [ ]:
# ============================== 10b. 5 TRUY VẤN THÀNH CÔNG + 5 THẤT BẠI (văn bản → ảnh) — luật chọn cố định ==============================
# Luật (CASE_ANALYSIS_LEVEL1_VI.md): seed 0, P-CHECK; thành công = 5 truy vấn tăng hạng nhiều nhất khi thêm đồ thị (hạng CLIP − hạng CLIP+đồ thị),
# thất bại = 5 truy vấn giảm nhiều nhất; hoà xếp theo id ảnh. Mỗi hình: ảnh đúng (kèm hạng trước/sau) + top-3 sau xếp hạng lại kèm bằng chứng đồ thị.
_ck, _rk = EXP1_KEEP["check"], EXP1_KEEP["ranks"]
_d = _rk["CLIP"] - _rk["G-full"]
_ord_up = sorted(range(len(_d)), key=lambda q: (-_d[q], _ck["ids"][q]))[:5]
_ord_dn = sorted(range(len(_d)), key=lambda q: (_d[q], _ck["ids"][q]))[:5]
CASES1 = []
for tag, L in (("thanh_cong", _ord_up), ("that_bai", _ord_dn)):
    for q in L:
        E, T = _ck["parsed"][q]
        order = np.lexsort((np.arange(_ck["topk"].shape[1]), -EXP1_KEEP["fused"]["G-full"][q]))[:3]
        top = []
        for kk in order:
            i = _ck["ids"][_ck["topk"][q, kk]]; dd = match_detail(T, E, GRAPH[i])
            top.append(dict(image_id=i, clip_pos=int(kk) + 1, relevant=bool(i == _ck["ids"][q]), edges=dd["edges"], nodes=dd["nodes"]))
        drel = match_detail(T, E, GRAPH[_ck["ids"][q]])
        CASES1.append(dict(kind=tag, query_image_id=_ck["ids"][q], caption=_ck["caps"][q], rank_clip=int(_rk["CLIP"][q]), rank_graph=int(_rk["G-full"][q]),
                           rank_obj_only=int(_rk["G-obj"][q]), parsed_triples=sorted("|".join(t) for t in T), parsed_entities=sorted(E),
                           relevant_evidence=dict(edges=drel["edges"], nodes=drel["nodes"]), top3=top))
        fig, axs = plt.subplots(1, 4, figsize=(13, 3.9))
        panels = [(_ck["ids"][q], f"ẢNH ĐÚNG: hạng CLIP {CASES1[-1]['rank_clip']} → {CASES1[-1]['rank_graph']}\nkhớp: {drel['nodes']} {[e.split('<=')[0] for e in drel['edges']]}")]
        panels += [(t["image_id"], f"#{n + 1} sau xếp hạng (CLIP #{t['clip_pos']}){' ✓' if t['relevant'] else ''}\nkhớp: {t['nodes']} {[e.split('<=')[0] for e in t['edges']]}")
                   for n, t in enumerate(top)]
        for ax, (i, title) in zip(axs, panels):
            ax.imshow(pil_path(VG_JPG[i])); ax.axis("off"); ax.set_title(title[:110], fontsize=7)
        fig.suptitle(f"[{tag}] '{_ck['caps'][q]}' | bộ ba tách được: {CASES1[-1]['parsed_triples']}", fontsize=8); fig.tight_layout()
        fig.savefig(f"{FIG}/case_{tag}_{len(CASES1):02d}_{_ck['ids'][q]}.png", dpi=95); plt.close(fig)
json.dump(CASES1, open(f"{OUT}/cases_text_to_image.json", "w"), indent=1, ensure_ascii=False, default=str)
p3_log(f"Ca văn bản → ảnh: {[(c['kind'], c['rank_clip'], c['rank_graph']) for c in CASES1]} | tăng {int((_d > 0).sum())}, giảm {int((_d < 0).sum())}, "
    f"không đổi {int((_d == 0).sum())}; trùng hạng với bản chỉ đối tượng: {float(np.mean(_rk['G-obj'] == _rk['G-full'])):.3f}")


## 11a. Phân tích bổ sung: độ nhạy α, so sánh cách tạo cạnh, file split

In [ ]:
# ============================== 11a. PHÂN TÍCH BỔ SUNG: độ nhạy α · so sánh cách tạo cạnh · xuất file split · hiển thị hình ==============================
from IPython.display import Image as _IImg, Markdown as _IMd, display as _idisp


def show_png(path, width=900):
    try:
        _idisp(_IImg(filename=path, width=width))
    except Exception:
        pass


# ---- 11a.1 Độ nhạy theo α (đề cương mục 7: "phân tích độ nhạy theo trọng số kết hợp α"), đo trên tập TUNE (validation) ----
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, res, key in (("TN1 luật (G-full)", RES1, None), ("TN1b GraphSAGE", RES1G, "SAGE"), ("TN1c SCENIR", RES1S, "SCENIR")):
    M = np.array([[(res["seeds"][s]["alpha_curve_tune_MRR"]["G-full"] if key is None else res["seeds"][s]["train"][key]["alpha_curve_tune_MRR"])[a]
                   for a in ALPHAS] for s in SEEDS])
    axs[0].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=name)
axs[0].set_title("TN1 văn bản → ảnh: MRR trên P-TUNE theo α", fontsize=9); axs[0].set_xlabel("α (1 = chỉ CLIP)"); axs[0].set_ylabel("MRR")
for g, res in (("ARO_all", RES2), ("ARO_geo", RES2), ("ARO_pred", RES2), ("SC_replace_rel", RES2), ("COCOSP_all", RES2B)):
    if g not in res["seeds"][0] or "alpha_grid_tune" not in res["seeds"][0][g]: continue
    M = np.array([[res["seeds"][s][g]["alpha_grid_tune"][a] for a in ALPHAS] for s in SEEDS]) * 100
    axs[1].errorbar(ALPHAS, M.mean(0), yerr=M.std(0, ddof=1), marker="o", ms=3, capsize=2, label=g)
axs[1].set_title("TN2/2b ảnh → caption: độ chính xác (%) trên TUNE theo α", fontsize=9); axs[1].set_xlabel("α (1 = chỉ CLIP)")
for ax in axs: ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIG}/alpha_sensitivity.png", dpi=120); plt.close(fig)
show_png(f"{FIG}/alpha_sensitivity.png")

# ---- 11a.2 So sánh các cách tạo cạnh / kiểu đường đi (đề cương mục 7, mở rộng) ----
EDGE_TABLE = [("TN1", "Không đồ thị (CLIP)", RES1["mean_std"]["CLIP"]["MRR"], "MRR"),
              ("TN1", "Chỉ đỉnh đối tượng (G-obj)", RES1["mean_std"]["G-obj"]["MRR"], "MRR"),
              ("TN1", "Bộ ba + cặp + đối tượng (G-full)", RES1["mean_std"]["G-full"]["MRR"], "MRR"),
              ("TN1", "G-full, vị từ hoán vị (G-shuf)", RES1["mean_std"]["G-shuf"]["MRR"], "MRR"),
              ("TN1", "G-full, đồ thị ảnh khác (G-rand)", RES1["mean_std"]["G-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE: ảnh – lớp – bộ ba", RES1G["mean_std"]["SAGE"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, cạnh ngẫu nhiên", RES1G["mean_std"]["SAGE-rand"]["MRR"], "MRR"),
              ("TN1b", "GraphSAGE, bỏ cạnh", RES1G["mean_std"]["SAGE-0"]["MRR"], "MRR"),
              ("TN1c", "SCENIR: scene graph lớp", RES1S["mean_std"]["SCENIR"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, cạnh ngẫu nhiên", RES1S["mean_std"]["SCENIR-rand"]["MRR"], "MRR"),
              ("TN1c", "SCENIR, bỏ cạnh", RES1S["mean_std"]["SCENIR-noedge"]["MRR"], "MRR")]
for g, res, lab in (("ARO_geo", RES2, "ARO trái/phải"), ("ARO_pred", RES2, "ARO vị từ"), ("COCOSP_all", RES2B, "COCO-spatial")):
    v = res["mean_std"][g]
    EDGE_TABLE += [(lab, "Cạnh vị từ / hình học (RelTR)", v["graph_only[dg]"], "acc"), (lab, "Đồ thị của mục khác", v["graph_only[dg_rand]"], "acc")]
    for m, d in (("graph_only[dg_nogeo]", "Bỏ vị trí (cạnh hình học = hằng)"), ("graph_only[dg_shuf]", "Vị từ hoán vị")):
        if m in v: EDGE_TABLE.append((lab, d, v[m], "acc"))
_md = ["| Thí nghiệm | Cách tạo cạnh / biến thể | Giá trị (trung bình ± std, 3 seed) |", "|---|---|---|"]
_md += [f"| {e} | {d} | {v['mean']:.4f} ± {v['std']:.4f} |" if u == "MRR" else f"| {e} | {d} | {100 * v['mean']:.1f}% ± {100 * v['std']:.1f} |" for e, d, v, u in EDGE_TABLE]
EDGE_MD = "\n".join(_md)
_idisp(_IMd("**So sánh các cách tạo cạnh** (TN1: MRR trên P-CHECK; TN2/2b: độ chính xác 'chỉ đồ thị' trên CHECK)\n\n" + EDGE_MD))

# ---- 11a.3 Xuất file split cố định (sản phẩm nộp: "tập chia dữ liệu cố định hoặc mã tạo split") ----
SPLIT_DIR = f"{OUT}/splits"; os.makedirs(SPLIT_DIR, exist_ok=True)
with open(f"{SPLIT_DIR}/tn1_vg_coco_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["vg_image_id", "coco_id", "split", "role"])
    for i in ELIG1: w.writerow([i, COCO_OF_VG[i], p5_split(i), "eval (P-CHECK)" if p5_split(i) == "P-CHECK" else "tune (P-TUNE) + GNN train"])
    for i in TRAIN_IMGS:
        if i not in set(ELIG1): w.writerow([i, "", "GNN-train-only", "GraphSAGE / SCENIR train"])
with open(f"{SPLIT_DIR}/tn2_tn2b_items_split.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["bench", "item", "image_key", "split_seed0", "split_seed1", "split_seed2"])
    for it in ITEMS2 + ITEMS2B: w.writerow([it["bench"], it["item"], it["key"]] + [v6_split(it["key"], s) for s in SEEDS])
json.dump(dict(rules={"VAL (v3)": "sha1(id) theo v3, chỉ số lọc ≥ 83000; heldout KHÔNG dùng",
                      "DEV-G0": "sha1('p3-dir2:'+id) mod 1000 < 350, trong pool G0 (VAL trừ benchmark v3, cache v4, 99 ảnh Phase 5)",
                      "TN1 P-TUNE/P-CHECK": "sha1('p5-pilot:'+id) mod 2", "TN2/2b TUNE/CHECK": "sha1(f'v6-{seed}:{image_key}') mod 2, seed ∈ {0,1,2}",
                      "lọc rò rỉ benchmark": "ảnh VG phải thuộc VAL; ảnh COCO phải không thuộc VG hoặc thuộc VG-VAL"},
               md5=dict(VAL=VAL_MD5, G0=G0_MD5), counts=dict(tn1=len(ELIG1), gnn_train=len(TRAIN_IMGS), tn2=len(ITEMS2), tn2b=len(ITEMS2B))),
          open(f"{SPLIT_DIR}/SPLIT_RULES.json", "w"), indent=1, ensure_ascii=False)
p3_log(f"Đã xuất file split vào {SPLIT_DIR}")

# ---- 11a.4 Hình chính (hiển thị trong notebook) ----
for p in [f"{FIG}/demo_1.png"] + sorted(glob.glob(f"{FIG}/case_*.png"))[:2] + sorted(glob.glob(f"{FIG}/xai_ex*.png"))[:2]:
    show_png(p, width=800)


## 11. Tổng hợp và đối chiếu yêu cầu đề cương

In [ ]:
# ============================== 11. TỔNG HỢP: bảng kết quả · đối chiếu tái lập với các lần chạy trước · lưu đầu ra ==============================
def pct(x): return f"{100 * x:.1f}"


lines = ["# Kết quả pipeline Project 3", "", f"Chế độ: {'SMOKE (dữ liệu giả — KHÔNG phải kết quả)' if SMOKE else 'REAL'} · {MODEL_INFO} · {elapsed()}", "",
         "## TN1 — Văn bản → ảnh, full-gallery (P-CHECK, trung bình ± std qua 3 seed caption)", "",
         "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1['decision']}**", "",
          "## TN1b — GraphSAGE (GNN đã học) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed)", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1G["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1G['decision']}** (SAGE-rand = cạnh ngẫu nhiên, SAGE-0 = bỏ cạnh)", "",
          f"## TN1c — SCENIR (ICML 2025) trong cùng pipeline văn bản → ảnh (P-CHECK, 3 seed; đặc trưng đỉnh: {RES1S['node_features']})", "",
          "| Phương pháp | R@1 | R@5 | R@10 | MRR |", "|---|---|---|---|---|"]
for k, v in RES1S["mean_std"].items():
    lines.append(f"| {k} | {pct(v['R1']['mean'])} ± {pct(v['R1']['std'])} | {pct(v['R5']['mean'])} | {pct(v['R10']['mean'])} | {v['MRR']['mean']:.4f} ± {v['MRR']['std']:.4f} |")
lines += ["", f"Quyết định đăng ký trước: **{RES1S['decision']}**", "",
          "## TN2 — Benchmark kết hợp (độ chính xác CHECK %, trung bình ± std qua 3 seed chia)", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
lines += ["", f"Quyết định đăng ký trước: **{RES2['decision']}**", "",
          f"## TN2b — Cấp 4 (a): cạnh hình học trên COCO-spatial ({RES2B['n_items']} mục sau lọc, {RES2B['n_scorable']} mục đồ thị chấm được; loại {RES2B['excluded']})", "",
          "| Nhóm | CLIP | Chỉ đồ thị | CLIP + đồ thị | Đồ thị mục khác | Bỏ vị trí |", "|---|---|---|---|---|---|"]
for g, v in RES2B["mean_std"].items():
    f = lambda m: f"{pct(v[m]['mean'])} ± {pct(v[m]['std'])}" if m in v else "—"
    lines.append(f"| {g} | {f('CLIP')} | {f('graph_only[dg]')} | {f('fusion[dg]')} | {f('graph_only[dg_rand]')} | {f('graph_only[dg_nogeo]')} |")
_cf = RES2B.get("counterfactual")
lines += ["", f"Quyết định đăng ký trước: **{RES2B['decision']}**" + (f" · Lật ảnh: đồ thị đảo {pct(_cf['graph_reversed'])}%, CLIP đảo {pct(_cf['clip_reversed'])}% (n={_cf['n']})" if _cf else ""), "",
          "## XAI (ARO, CHECK seed 0)", "",
          "| Nhóm | n | Lật ảnh: đồ thị đảo / CLIP đảo | Xoá bằng chứng: đổi / ngẫu nhiên | Validity cả hai / ít nhất một | Sufficiency | Sparsity: kích thước / độ tập trung |",
          "|---|---|---|---|---|---|---|"]
for k, o in RES3.items():
    cf = f"{pct(o['counterfactual_graph_reversed'])} / {pct(o['counterfactual_clip_reversed'])}" if "counterfactual_graph_reversed" in o else "—"
    lines.append(f"| {k} | {o['n']} | {cf} | {pct(o.get('fidelity_del_changed', 0))} / {pct(o.get('fidelity_rand_changed', 0))} | "
                 f"{pct(o['validity_both_iou50'] or 0)} / {pct(o['validity_any_iou50'] or 0)} | {pct(o.get('sufficiency_kept', 0))} (theo cấu trúc) | "
                 f"{pct(o.get('sparsity_evidence_fraction', 0))}% (thiết kế) / {o.get('sparsity_concentration_mean', 0):.2f} (trung vị {o.get('sparsity_concentration_median', 0):.2f}) |")

# ---- đối chiếu tái lập với số đã chạy riêng (v5, v6, v6-XAI) ----
REPRO = []
if not SMOKE:
    for k, ref in REF["exp1_MRR"].items():
        REPRO.append(("TN1 MRR " + k, ref, RES1["mean_std"][k]["MRR"]["mean"]))
    for g, d in REF["exp2_acc"].items():
        for m, ref in d.items(): REPRO.append((f"TN2 {g} {m}", ref, RES2["mean_std"][g][m]["mean"]))
    for k, d in REF["xai"].items():
        for m, ref in d.items(): REPRO.append((f"XAI {k} {m}", ref, RES3[k][m]))
    lines += ["", "## Đối chiếu tái lập với các lần chạy riêng trước đó", "", "| Chỉ số | Lần trước | Pipeline này | Chênh |", "|---|---|---|---|"]
    lines += [f"| {n} | {a:.4f} | {b:.4f} | {b - a:+.4f} |" for n, a, b in REPRO]
lines += ["", "## So sánh các cách tạo cạnh", "", EDGE_MD]

# ---- Cấp 4 (b): video MSVD (notebook video v5 chạy nguyên vẹn ở mục 4v) ----
lines += ["", f"## Cấp 4 (b) — Video MSVD (notebook Level4B v5, trạng thái: {RES_VIDEO['status']}, {RES_VIDEO.get('minutes')} phút)", ""]
if RES_VIDEO["status"] == "ok":
    lines += [f"Lần nhìn test MSVD: **{RES_VIDEO.get('test_looks_msvd')}** — số test chỉ là thăm dò, không phải xác nhận độc lập.", "",
              "| Mô hình | Tập | R@1 | R@5 | R@10 | MRR | n truy vấn |", "|---|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r.get('set')} | {pct(r['R@1'])} | {pct(r['R@5'])} | {pct(r['R@10'])} | {r['MRR']:.4f} | {r.get('n_queries')} |" for r in RES_VIDEO["test"]]
    if RES_VIDEO["comparisons"]:
        lines += ["", "| So sánh | a vs b | ΔMRR [CI 95%] | p (Holm) |", "|---|---|---|---|"]
        lines += [f"| {c.get('comparison')} | {c.get('a')} vs {c.get('b')} | {c.get('diff', 0):+.4f} [{c.get('ci_lo', 0):+.4f}, {c.get('ci_hi', 0):+.4f}] | {c.get('p_holm')} |"
                  for c in RES_VIDEO["comparisons"]]
    if RES_VIDEO.get("fidelity"): lines += ["", f"Fidelity (video): {RES_VIDEO['fidelity']}"]
else:
    lines += [f"Nhánh video không hoàn tất: {RES_VIDEO.get('error', '')[:500]}"]

# ---- bảng đối chiếu yêu cầu đề cương (sinh tự động từ kết quả của chính lần chạy này) ----
_d = lambda dec: "DƯƠNG" if (dec == "GO" or (isinstance(dec, str) and dec.startswith("DƯƠNG"))) else "không dương"
CHECKLIST = [
    ("Cấp 1", "Baseline CLIP đóng băng trên ≥ 1 tập công khai", "✅", f"CLIP {CLIP_MODEL}; TN1 MRR {RES1['mean_std']['CLIP']['MRR']['mean']:.4f}"),
    ("Cấp 1", "Đồ thị từ dữ liệu có sẵn (scene graph VG)", "✅", "Scene graph GT (TN1), scene graph RelTR (TN2, 2b)"),
    ("Cấp 1", "≥ 1 thuật toán đã học (GraphSAGE)", "✅", f"TN1b: SAGE {RES1G['mean_std']['SAGE']['MRR']['mean']:.4f} vs CLIP {RES1G['mean_std']['CLIP']['MRR']['mean']:.4f} ({_d(RES1G['decision']['G1 SAGE-CLIP'])})"),
    ("Cấp 1", "Xếp hạng lại α·cos + (1−α)·graph, so CLIP vs CLIP+Graph cùng split", "✅", f"TN1 {RES1['decision']}; ARO {RES2['decision'].get('ARO_all')}"),
    ("Cấp 1", "Ablation bỏ cạnh / cạnh ngẫu nhiên", "✅", f"ARO: mục khác {pct(RES2['mean_std']['ARO_all']['graph_only[dg_rand]']['mean'])}%, bỏ vị trí {pct(RES2['mean_std']['ARO_all']['graph_only[dg_nogeo]']['mean'])}%"),
    ("Cấp 1", "Hiển thị quan hệ/đường đi giải thích", "✅", "Demo search() + hình XAI"),
    ("Cấp 1", "5 truy vấn thành công + 5 thất bại", "✅", f"{len(CASES1)} ca văn bản → ảnh (case_*.png) + 10 ca ARO (xai_ex*.png); phân tích: CASE_ANALYSIS_LEVEL1_VI.md"),
    ("Cấp 1", "R@1/5/10, MRR, ≥ 3 seed, α chọn trên validation", "✅", "Mọi thí nghiệm: 3 seed, α trên TUNE"),
    ("Cấp 2", "≥ 2 tập công khai", "✅", "Visual Genome + COCO, ARO, SugarCrepe, COCO-spatial"),
    ("Cấp 2", "1 phương pháp mới từ 2024, cùng pipeline/split", "✅", f"SCENIR (ICML 2025) TN1c: {RES1S['mean_std']['SCENIR']['MRR']['mean']:.4f} vs CLIP ({_d(RES1S['decision']['S1 SCENIR-CLIP'])})"),
    ("Cấp 2", "Bộ ≥ 20 truy vấn có nhóm gián tiếp, báo cáo tách nhóm", "✅", "Ở phần A (eval_queries_20.json, tách direct/indirect) và MMKG (mmkg/)"),
    ("Cấp 2", "Ablation xáo trộn / đồ thị ngẫu nhiên; so sánh cách tạo cạnh; độ nhạy α", "✅", "Bảng so sánh cạnh + alpha_sensitivity.png"),
    ("Cấp 3", "Fidelity định lượng ≥ 10 truy vấn", "✅", f"Lật ảnh {pct(RES3['all'].get('counterfactual_graph_reversed', 0))}% vs CLIP {pct(RES3['all'].get('counterfactual_clip_reversed', 0))}%; xoá {pct(RES3['all']['fidelity_del_changed'])}% vs ngẫu nhiên {pct(RES3['all']['fidelity_rand_changed'])}% (n={RES3['all']['n']})"),
    ("Cấp 3", "Validity", "✅", f"Khớp GT IoU≥0.5: cả hai {pct(RES3['all']['validity_both_iou50'] or 0)}%, ít nhất một {pct(RES3['all']['validity_any_iou50'] or 0)}%"),
    ("Cấp 3", "Sparsity", "✅", f"Kích thước {pct(RES3['all']['sparsity_evidence_fraction'])}% (thiết kế); độ tập trung trung bình {RES3['all']['sparsity_concentration_mean']:.2f}"),
    ("Cấp 3", "MMKG tự xây + data card", "✅", "Ở mmkg/: Project3_MMKG_VN.ipynb + DATA_CARD.md"),
    ("Cấp 4 (a)", "Cải tiến của nhóm + giả thuyết + ablation + ≥ 2 tập công khai", "✅" if RES2B["decision"]["A1 fusion-CLIP"].startswith("DƯƠNG") else "🟡",
     f"Cạnh hình học: ARO trái/phải {pct(RES2['mean_std']['ARO_geo']['CLIP']['mean'])}→{pct(RES2['mean_std']['ARO_geo']['fusion[dg]']['mean'])}%; COCO-spatial {RES2B['decision']}"),
    ("Cấp 4 (b)", "Mở rộng video (text-to-video)", "✅" if RES_VIDEO["status"] == "ok" else "❌",
     (lambda tv: f"MSVD test: CLIP {tv['M0_clip']['MRR']:.4f} → đồ thị đầy đủ {tv['M4_full_graph']['MRR']:.4f} (lần nhìn test: {RES_VIDEO.get('test_looks_msvd')})"
      if RES_VIDEO["status"] == "ok" and "M0_clip" in tv and "M4_full_graph" in tv else f"trạng thái {RES_VIDEO['status']}")(
         {r["model"]: r for r in RES_VIDEO.get("test", []) if r.get("set") == "test_all"})),
    ("Mở rộng", "Chiều ảnh → văn bản (nhánh B)", "✅", "TN2, TN2b"),
    ("Nộp", "Mã tạo split / file split cố định", "✅", "splits/ (CSV + SPLIT_RULES.json)"),
    ("Nộp", "Notebook nhập truy vấn → top-k + giải thích", "✅", "Hàm search() (mục 10)")]
lines += ["", "## Đối chiếu yêu cầu đề cương", "", "| Cấp | Yêu cầu | Trạng thái | Bằng chứng (lần chạy này) |", "|---|---|---|---|"]
lines += [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in CHECKLIST]
REPORT = "\n".join(lines)
open(f"{OUT}/RESULTS.md", "w", encoding="utf-8").write(REPORT)
json.dump(dict(exp1=RES1, exp1_graphsage=RES1G, exp1_scenir=RES1S, exp2=RES2, exp2b_cocospatial=RES2B, xai=RES3, video=RES_VIDEO, demo=DEMO, checklist=CHECKLIST, models=MODEL_INFO, smoke=SMOKE, inputs_sha256=SHA, bench_sha256=BSHA,
               reproduction=[dict(metric=n, previous=a, now=b) for n, a, b in REPRO], runtime_s=round(time.time() - T0, 1)),
          open(f"{OUT}/summary.json", "w"), indent=1, default=str, ensure_ascii=False)
for name, rows in (("exp2_items.csv", ROWS2), ("exp2b_cocospatial_items.csv", ROWS2B), ("xai_items.csv", ROWS3)):
    with open(f"{OUT}/{name}", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r})); w.writeheader(); w.writerows(rows)
try:
    _idisp(_IMd(REPORT))
except Exception:
    print(REPORT)
p3_log("Xong toàn bộ pipeline:", elapsed())


In [ ]:
# ============ TỔNG KẾT: nơi lưu kết quả của cả hai phần ============
import os, glob
for _p in ["/kaggle/working/results", "/kaggle/working/output/action_genome/results", "/kaggle/working/p3_out", "/kaggle/working/p3_out/video_v2"]:
    _fs = sorted(glob.glob(os.path.join(_p, "*")))
    print(f"{_p}: {len(_fs)} tệp")
    for _f in _fs[:40]:
        print("   ", os.path.basename(_f))

print("Phần C (Level 4(b) v2-graphfix):", {k: v for k, v in RES_L4B_V2.items() if k != "error"} if "RES_L4B_V2" in globals() else "chưa chạy")
